<a id="top"></a>

# Moving a Bell pair across three processor graphs, and what error detection buys

**bell-transport-bench · UCF IBM Qiskit Fall Fest 2026 · quanTA challenge**

**Abstract.** We built a benchmark that creates the Bell state |Φ+⟩ between two fixed qubits, the ports, on three processor graphs, under one noise model built from IBM's Boston calibration, and measures what parity checks with postselection buy and cost. Moving the pair with a chain of two-CX moves instead of the organisers' SWAPs out and back raised the fidelity F on all three graphs, from 0.9770 to 0.9805 on the 2016-inspired graph, 0.9614 to 0.9756 on the contemporary graph and 0.9681 to 0.9788 on the hyperbolic-inspired graph (each ± 0.0007 to 0.0010, one standard error). An XX parity check read at the end of the circuit on a spare path raised F further on the contemporary graph, to 0.9793 ± 0.0007 keeping 97.5% of shots, and on the hyperbolic-inspired graph, to 0.9812 ± 0.0007 keeping 98.3%. The 2016-inspired graph has no spare path, so its check reads a qubit mid-circuit, and it did not beat the move chain alone: 0.9786 ± 0.0007 keeping 97.0%, against 0.9805 ± 0.0007. Every result holds for this one noise model; appendix A shows how far it holds when the model changes. [check]

**To run it:** open it in Colab or Jupyter and run all cells. It needs no download beyond the pinned Python packages: the code and the recorded runs come as two payloads from the repository at its tag, and every config file is written out where the notebook explains it.

| § | Section | Challenge material |
|---|---|---|
| [1](#s1) | Setup | 01 Reproducible notebook |
| [2](#s2) | The task | 01 (assumptions) |
| [3](#s3) | Methods: [graphs](#s3-1), [noise](#s3-2), [circuits](#s3-3), [benchmark files](#s3-4), [transpiled circuits](#s3-5), [metrics](#s3-6), [the transpiler](#s3-7) | 02 Graphs and circuits; 01 (assumptions) |
| [4](#s4) | Results: [nine cases](#s4-1), [protection](#s4-2), [architecture](#s4-3), [rerun a case](#s4-4) | 03 Quantitative comparison |
| [5](#s5) | Interpretation | 04 Scientific interpretation |
| [6](#s6) | Limitations | 04 (limitations) |
| [7](#s7) | Recommendation | 05 Hardware or software recommendation |
| [A](#sA) | Appendix A. Optional studies: [mid-circuit error](#sA-1), [starting excitation](#sA-2), [middle out](#sA-3), [vacancy read](#sA-4) | 04 (how far the results hold) |
| [B](#sB) | Appendix B. The studies' files | 01 |
| [C](#sC) | Appendix C. Config files beyond the Boston runs | 01 |
| [D](#sD) | Appendix D. Sources and AI use | 01 |

<a id="s1"></a>

## 1. Setup

**Run the cells of this section first.** The first two cells are payloads, each one line, from the repository [`Rickory2021/bell-transport-bench`](https://github.com/Rickory2021/bell-transport-bench/tree/submission) at tag `submission`:

- **The code:** the `bellbench` package, `src/bellbench/`, exactly as the repository holds it (19 files).
- **The recorded runs:** from `results/runs/benchmark/`, every file of the snapshot behind the nine cases, `move-chain/20261116T213408Z`, and each study's table, report and run list (51 files).

The third cell installs the pinned packages where they are missing, unpacks both payloads into a temporary folder, and lists every file with its SHA-256, so each one can be checked against the same file at that tag. Nothing is written outside that folder.

**Every config file is written out where this notebook explains it,** not packed: each `%%repo_file` cell writes one file into the same folder, and the code reads it from there. The schemas come first, since every config file and run record is checked against them when it loads.

In [ ]:
CODE = "..."

In [ ]:
RESULTS = "..."

In [ ]:
import base64, hashlib, html, io, subprocess, sys, tempfile, zipfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from IPython.core.magic import register_cell_magic
from IPython.display import HTML, display

PINNED = ["qiskit==2.6.2", "qiskit-aer==0.18.2", "qiskit-ibm-runtime==0.51.0", "qiskit-qasm3-import==0.6.0",
          "openqasm3[parser]==1.0.1", "networkx==3.7", "pylatexenc==2.10", "jsonschema", "pyyaml", "pandas",
          "matplotlib"]


def _present(requirement):
    name, _, wanted = requirement.partition("==")
    try:
        found = version(name.split("[")[0])
    except PackageNotFoundError:
        return False
    return not wanted or found == wanted


missing = [r for r in PINNED if not _present(r)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

WORK = Path(tempfile.mkdtemp(prefix="bellbench-"))
manifests = []
for title, payload in (("The code payload", CODE), ("The recorded-runs payload", RESULTS)):
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
        archive.extractall(WORK)
        rows = "".join(f"<tr><td><code>{html.escape(n)}</code></td>"
                       f"<td><code>{hashlib.sha256(archive.read(n)).hexdigest()}</code></td></tr>"
                       for n in archive.namelist())
        manifests.append(f"<details><summary><b>{title}: {len(archive.namelist())} files, each with its SHA-256</b>"
                         f"</summary><table><tr><th>file</th><th>SHA-256</th></tr>{rows}</table></details>")
sys.path.insert(0, str(WORK / "src"))
from bellbench import configs  # reads every config file from WORK


@register_cell_magic
def repo_file(line, cell):
    """Write the cell's text to the repository path on the magic's line, inside WORK, and show its content hash."""
    path = WORK / line.strip()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(cell, encoding="utf-8")
    print(f"wrote {line.strip()} · content hash {configs.content_hash(path)[:19]}…")


display(HTML("".join(manifests)))
print(f"installed {len(missing)} packages; unpacked both payloads into {WORK}")

**The schemas:** what a valid processor file, noise profile, circuit file, benchmark file, run list and run record must hold.

In [ ]:
%%repo_file config/bp.schema.benchmark.yaml
# config/bp.schema.benchmark.yaml
#
# Validates config/benchmark/bp.data.benchmark.*.yaml: one benchmark, a named set
# of runs with one comparison table across them (docs/DESIGN.md, section 6). Each
# run is a circuit file under a noise profile; for the core benchmark, cases map
# the challenge's nine cases onto a run and a simulation. Checks that span fields
# (case runs exist, ids are unique, each run and simulation fills one case, each
# case sits where the challenge's grid puts it) are in src/bellbench/run.py, which
# runs them whenever a benchmark file loads.

$schema: "https://json-schema.org/draft/2020-12/schema"
title: Benchmark file
description: The runs a benchmark makes, their shots and base seed, and which run fills which case.

type: object
required: [name, label, notes, shots, base_seed, runs]
additionalProperties: false

$defs:
  text:
    type: string
    minLength: 1
  slug:
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'

properties:
  name:
    description: Slug, the qualifier in the file name; make benchmark NAME=<name> runs it.
    $ref: "#/$defs/slug"
  label: {$ref: "#/$defs/text"}
  notes:
    description: Free text saying what the benchmark answers and anything a reader should know.
    $ref: "#/$defs/text"
  shots:
    description: Shots per basis circuit in each simulation of every run.
    type: integer
    minimum: 1
  base_seed:
    description: The base seed every run derives its seeds from.
    type: integer
    minimum: 0
    maximum: 4294967295
  runs:
    description: The runs, each a circuit file under a noise profile, with any scale, overrides and conditional channels.
    type: array
    minItems: 1
    items:
      type: object
      required: [run, circuit, noise]
      additionalProperties: false
      properties:
        run:
          description: >-
            The run's name within this benchmark, unique in the file: its folder in the benchmark's results, the
            run column of the comparison table, and the run a case names.
          $ref: "#/$defs/slug"
        circuit: {$ref: "#/$defs/slug"}
        noise: {$ref: "#/$defs/slug"}
        scale: {type: number, exclusiveMinimum: 0}
        overrides:
          type: array
          items:
            type: object
            required: [qubits, field, value]
            additionalProperties: false
            properties:
              qubits:
                oneOf:
                  - {type: integer, minimum: 0}
                  - {type: array, minItems: 1, maxItems: 2, items: {type: integer, minimum: 0}}
              field: {$ref: "#/$defs/text"}
              value: {type: number}
        conditional:
          type: object
          additionalProperties: false
          properties:
            reset_error: {type: [number, "null"]}
            mid_circuit_measurement_error: {type: [number, "null"]}
            starting_excitation: {type: [number, "null"]}
  cases:
    description: >-
      For a benchmark that fills the challenge's nine cases (docs/CHALLENGE.md, The nine core cases), which run and
      simulation fills each one; a benchmark without cases is a study whose runs fill none.
    type: array
    items:
      type: object
      required: [case, run, simulation]
      additionalProperties: false
      properties:
        case:
          description: >-
            The case's number in the challenge's grid: 1 to 3 the 2016-inspired model, 4 to 6 the contemporary
            model, 7 to 9 the hyperbolic-inspired model, each as the ideal baseline, the noisy baseline and noisy
            with protection.
          type: integer
          minimum: 1
          maximum: 9
        run:
          description: The run, by its name in this file's runs.
          $ref: "#/$defs/slug"
        simulation:
          description: Which of that run's two simulations fills the case.
          enum: [ideal, noisy]

In [ ]:
%%repo_file config/bp.schema.circuit.yaml
# config/bp.schema.circuit.yaml
#
# Validates config/circuit/bp.data.circuit.*.yaml: one written circuit on one
# processor model's physical qubits, as a run reads it (docs/DESIGN.md, sections
# 2 and 5). The gates are OpenQASM 3, so a reader sees exactly what runs. A run
# adds each basis's rotation and the port measurement itself, into a classical
# register named port; a written circuit names its own registers by role.
# Checks that span fields (the gates parse, the circuit has the processor's
# qubits, the registers are named by role) are in src/bellbench/circuits.py,
# which runs them whenever a circuit file loads.

$schema: "https://json-schema.org/draft/2020-12/schema"
title: Circuit file
description: One written circuit on one processor model, with the analysis that processes its shots.

type: object
required: [name, label, processor, analysis, notes, circuit]
additionalProperties: false

$defs:
  text:
    type: string
    minLength: 1
  slug:
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'

properties:
  name:
    description: Slug, the qualifier in the file name; runs and benchmarks pick the circuit by it.
    $ref: "#/$defs/slug"
  label:
    description: A short human name for the circuit.
    $ref: "#/$defs/text"
  processor:
    description: The name of the processor file whose physical qubits the gates act on.
    $ref: "#/$defs/slug"
  analysis:
    description: >-
      How a run reads the shots: the name of an analysis in the registry in src/bellbench/analysis.py, which
      checks its settings when the file loads. postselect takes none; parity takes checks, frame_z and frame_x.
    type: object
    required: [name]
    additionalProperties: false
    properties:
      name: {$ref: "#/$defs/slug"}
      settings: {type: object}
  notes:
    description: Free text saying how the circuit was built (the transport, the path, the checks) and anything a reader should know.
    $ref: "#/$defs/text"
  circuit:
    description: >-
      The gates, as an OpenQASM 3 program on one register q with the processor's qubit count; classical registers
      named by role (check, flag, frame_z or frame_x), alone or followed by an underscore and a name.
    $ref: "#/$defs/text"

In [ ]:
%%repo_file config/bp.schema.noise.yaml
# config/bp.schema.noise.yaml
#
# Validates config/noise/bp.data.noise.*.yaml: one noise profile, NoiseControl's
# settings in Qiskit's terms (docs/DESIGN.md, section 4). A snapshot profile holds
# one value per quantity, the median over a pinned IBM fake-backend snapshot's
# qubits and directed couplers, and its own native gate set, so any profile pairs
# with any processor graph. A depolarising profile holds a single channel after a
# named gate, such as the organisers' cross-check, and a pauli profile a Pauli
# error with a chosen noise bias after a named gate, for the biased-noise study.
# Times are in seconds, errors are probabilities. Keys are descriptive; gate names are Qiskit's, since
# circuits compile to those instructions.
#
# Each value field's description is also the comment above that field in every
# noise file; tests/test_config.py fails when the two differ, so edit them here
# and copy them there. tests/test_config.py also pins every snapshot value to its
# snapshot.

$schema: "https://json-schema.org/draft/2020-12/schema"
title: Noise profile
description: The noise values NoiseControl builds a Target and a noise model from.

type: object
required: [name, label, kind, reference, notes]
additionalProperties: false

$defs:
  text:
    type: string
    minLength: 1
  seconds:
    type: number
    minimum: 0
  url:
    type: string
    pattern: '^https?://'
  probability:
    type: number
    minimum: 0
    maximum: 1
  gate:
    type: object
    required: [qubits, error_probability, duration]
    additionalProperties: false
    properties:
      qubits:
        description: 1 for a one-qubit gate, 2 for a two-qubit gate.
        enum: [1, 2]
      error_probability: {$ref: "#/$defs/probability"}
      duration: {$ref: "#/$defs/seconds"}
      from_snapshot_gate:
        description: The snapshot's own gate the values were taken from, when it differs from this gate's name.
        $ref: "#/$defs/text"
  timed:
    type: object
    required: [duration]
    additionalProperties: false
    properties:
      duration: {$ref: "#/$defs/seconds"}

properties:
  name:
    description: Slug, the qualifier in the file name; runs and benchmarks pick the profile by it.
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'
  label: {$ref: "#/$defs/text"}
  kind:
    description: >-
      snapshot, medians of a fake-backend snapshot; depolarising, one depolarising channel after a named gate;
      pauli, a Pauli error with a chosen noise bias on each qubit of a named gate.
    enum: [snapshot, depolarising, pauli]
  reference:
    description: The source's id in docs/bp.data.references.yaml.
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'
  notes:
    description: Free text naming the source in words, and anything a reader of the file should know.
    $ref: "#/$defs/text"
  snapshot:
    description: The pinned snapshot a snapshot profile's values come from, with links to its files at the pinned tag; tests read these fields.
    type: object
    required: [class, package, calibrated, properties, configuration]
    additionalProperties: false
    properties:
      class:
        description: The qiskit-ibm-runtime fake backend class.
        $ref: "#/$defs/text"
      package:
        description: The package version that pins the snapshot.
        $ref: "#/$defs/text"
      calibrated:
        description: The snapshot's calibration date.
        type: string
        pattern: '^\d{4}-\d{2}-\d{2}$'
      properties:
        description: Link to the snapshot's properties file (per-qubit and per-gate values) at the pinned package's tag.
        $ref: "#/$defs/url"
      configuration:
        description: Link to the snapshot's configuration file (the clock tick) at the pinned package's tag.
        $ref: "#/$defs/url"
  time_step:
    description: "Time step in seconds: the hardware's clock tick. Null when the snapshot records none."
    oneOf:
      - type: number
        exclusiveMinimum: 0
      - type: "null"
  qubit:
    type: object
    required: [t1_relaxation_time, t2_dephasing_time, readout_error]
    additionalProperties: false
    properties:
      t1_relaxation_time:
        description: "T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1)."
        $ref: "#/$defs/seconds"
      t2_dephasing_time:
        description: "T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1."
        $ref: "#/$defs/seconds"
      readout_error:
        description: Asymmetric readout errors, which replace Aer's symmetric ones.
        type: object
        required: [reads_1_when_0, reads_0_when_1]
        additionalProperties: false
        properties:
          reads_1_when_0:
            description: "Readout error: the probability of reading 1 when the qubit was 0."
            $ref: "#/$defs/probability"
          reads_0_when_1:
            description: "Readout error: the probability of reading 0 when the qubit was 1."
            $ref: "#/$defs/probability"
  gates:
    description: "Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds."
    type: object
    minProperties: 1
    additionalProperties: {$ref: "#/$defs/gate"}
  measure:
    description: "Measurement: its duration in seconds. Its errors are the readout errors."
    $ref: "#/$defs/timed"
  reset:
    description: "Reset to 0: its duration in seconds."
    $ref: "#/$defs/timed"
  borrowed:
    description: Values the snapshot lacks, taken from another source and marked as borrowed.
    type: array
    items:
      type: object
      required: [field, value, source, url, why]
      additionalProperties: false
      properties:
        field: {$ref: "#/$defs/text"}
        value: {type: number}
        source: {$ref: "#/$defs/text"}
        url:
          description: Link to the file the value comes from, at the pinned tag when it is a snapshot file.
          $ref: "#/$defs/url"
        why: {$ref: "#/$defs/text"}
  depolarising:
    type: object
    required: [after_gate, probability, gates]
    additionalProperties: false
    properties:
      after_gate:
        description: The gate each depolarising channel follows.
        $ref: "#/$defs/text"
      probability:
        description: "Depolarising strength: with this probability, the gate's qubits are left in a fully random state."
        $ref: "#/$defs/probability"
      gates:
        description: The gate set; every gate other than after_gate is ideal.
        type: array
        minItems: 1
        items: {$ref: "#/$defs/text"}
  pauli:
    type: object
    required: [after_gate, probability, bias, gates]
    additionalProperties: false
    properties:
      after_gate:
        description: The gate each Pauli error follows, on every qubit it acts on.
        $ref: "#/$defs/text"
      probability:
        description: "The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit."
        $ref: "#/$defs/probability"
      bias:
        description: "Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is."
        type: number
        exclusiveMinimum: 0
      gates:
        description: The gate set; every gate other than after_gate is ideal.
        type: array
        minItems: 1
        items: {$ref: "#/$defs/text"}

allOf:
  - if: {required: [kind], properties: {kind: {const: snapshot}}}
    then: {required: [snapshot, time_step, qubit, gates, measure]}
  - if: {required: [kind], properties: {kind: {const: depolarising}}}
    then: {required: [depolarising]}
  - if: {required: [kind], properties: {kind: {const: pauli}}}
    then: {required: [pauli]}

In [ ]:
%%repo_file config/bp.schema.processor.yaml
# config/bp.schema.processor.yaml
#
# Validates config/processor/bp.data.processor.*.yaml: one processor model, its
# qubits, its two ports and its coupling graph. The organisers' three graphs are
# the core set; any other graph is a further file of the same shape. Checks that
# span fields (ids in order, ports and edges naming real qubits, each edge once,
# positions for every qubit or none, a connected graph) are check_processor in
# src/bellbench/configs.py, which runs whenever a file loads; tests/test_config.py
# checks that the reference id exists.

$schema: "https://json-schema.org/draft/2020-12/schema"
title: Processor model
description: A processor's qubits, its two ports and the couplers between its qubits.

type: object
required: [name, label, reference, notes, qubits, ports, edges]
additionalProperties: false

$defs:
  text:
    type: string
    minLength: 1
  qubit_id:
    description: A qubit's id, Qiskit's index for that physical qubit.
    type: integer
    minimum: 0

properties:
  name:
    description: Slug, the qualifier in the file name; runs and benchmarks pick the processor by it.
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'
  label:
    description: The model's name as the challenge gives it.
    $ref: "#/$defs/text"
  reference:
    description: The source's id in docs/bp.data.references.yaml.
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'
  notes:
    description: Free text naming the source in words, and anything a reader of the file should know.
    $ref: "#/$defs/text"
  qubits:
    description: One entry per qubit, in id order; the qubit count is the list's length.
    type: array
    minItems: 2
    items:
      type: object
      required: [id]
      additionalProperties: false
      properties:
        id:
          description: Qiskit's index for this physical qubit, 0 to n - 1, each once.
          $ref: "#/$defs/qubit_id"
        position:
          description: Drawing coordinates (x, y), used only for figures. Give them for every qubit or for none; without them, figures use an automatic layout.
          type: array
          minItems: 2
          maxItems: 2
          items: {type: number}
  ports:
    description: The ids of the two qubits where the Bell pair must end up. Their positions are the requirement; the ids are the source's labels.
    type: array
    minItems: 2
    maxItems: 2
    items: {$ref: "#/$defs/qubit_id"}
  edges:
    description: Undirected couplers between qubit ids; a two-qubit gate may act on each pair in either direction.
    type: array
    minItems: 1
    uniqueItems: true
    items:
      type: array
      minItems: 2
      maxItems: 2
      items: {$ref: "#/$defs/qubit_id"}

In [ ]:
%%repo_file config/bp.schema.result.yaml
# config/bp.schema.result.yaml
#
# Validates bp.data.result.yaml in each run folder, results/runs/... or
# workspace/runs/...: the settings and records of one run, which CircuitRun
# writes and a rerun rebuilds from (docs/DESIGN.md, section 5). The numbers are
# in the folder's simulations.csv and counts.csv, never here. CircuitRun checks
# a record against this schema when it writes one and when it reads one back.

$schema: "https://json-schema.org/draft/2020-12/schema"
title: Run record
description: One run's files, settings, seeds, circuits, cost and provenance.

type: object
required: [result_format, circuit, noise_control, shots, base_seed, seeds, diagrams, registers, written, compiled, cost, scheduled_duration, simulations, provenance]
additionalProperties: false

$defs:
  text:
    type: string
    minLength: 1
  file:
    description: A config file by its repo-relative path and the hash of its parsed content.
    type: object
    required: [path, hash]
    additionalProperties: false
    properties:
      path: {$ref: "#/$defs/text"}
      hash: {type: string, pattern: '^sha256:[0-9a-f]{64}$'}
  seed:
    type: integer
    minimum: 0
    maximum: 4294967295
  bases:
    description: One entry for each of the three bases.
    type: object
    required: [XX, YY, ZZ]
    propertyNames: {enum: [XX, YY, ZZ]}
  simulation:
    type: object
    required: [seeds, methods]
    additionalProperties: false
    properties:
      seeds:
        description: The simulator seed Aer used for each basis circuit.
        allOf: [{$ref: "#/$defs/bases"}]
        additionalProperties: {$ref: "#/$defs/seed"}
      methods:
        description: The simulation method Aer used for each basis circuit.
        allOf: [{$ref: "#/$defs/bases"}]
        additionalProperties: {$ref: "#/$defs/text"}
      empty_bases:
        description: The bases in which the simulation kept no shot, so its F is undefined; written only if any.
        type: array
        minItems: 1
        uniqueItems: true
        items: {enum: [XX, YY, ZZ]}

properties:
  result_format:
    description: >-
      The shape of this record; benchmarks gather only runs of the current format. 1: circuits in OpenQASM 2 and
      a counts.csv column per register role; 2: circuits in OpenQASM 3 and counts.csv's full outcome key. A record
      of an earlier format stays valid as history; reruns and views refuse it.
    enum: [1, 2]
  circuit: {$ref: "#/$defs/file"}
  noise_control:
    description: NoiseControl's export, taken when it built the model.
    type: object
    required: [processor, noise, scale, overrides, conditional]
    additionalProperties: false
    properties:
      processor: {$ref: "#/$defs/file"}
      noise: {$ref: "#/$defs/file"}
      scale: {type: number, exclusiveMinimum: 0}
      overrides:
        type: array
        items:
          type: object
          required: [qubits, field, value]
          additionalProperties: false
          properties:
            qubits: {type: array, minItems: 1, maxItems: 2, items: {type: integer, minimum: 0}}
            field: {$ref: "#/$defs/text"}
            value: {type: number}
      conditional:
        type: object
        required: [reset_error, mid_circuit_measurement_error]
        additionalProperties: false
        properties:
          reset_error: {type: [number, "null"]}
          mid_circuit_measurement_error: {type: [number, "null"]}
          starting_excitation:
            description: The probability that each qubit starts a shot in |1>; recorded only when set, absent meaning none.
            type: number
  shots:
    description: Shots per basis circuit in each simulation.
    type: integer
    minimum: 1
  base_seed: {$ref: "#/$defs/seed"}
  seeds:
    description: The seeds the run derived from its base seed and labels, one per step.
    type: object
    required: [transpile, ideal, noisy]
    additionalProperties: false
    properties:
      transpile: {$ref: "#/$defs/seed"}
      ideal: {$ref: "#/$defs/seed"}
      noisy: {$ref: "#/$defs/seed"}
  diagrams:
    description: Recorded as the run was given it. It no longer chooses a run folder's pictures, which are the same for every run (docs/DESIGN.md, section 5).
    enum: [lean, full]
  registers:
    description: >-
      The compiled circuits' classical registers in order, the port register last; counts.csv's outcome key holds
      one bit string per register, in reverse of this order.
    type: array
    minItems: 1
    items: {$ref: "#/$defs/text"}
  written:
    description: The circuit as written, in OpenQASM 3.
    $ref: "#/$defs/text"
  compiled:
    description: Each basis circuit as compiled, in OpenQASM 3 on physical qubits, before scheduling.
    allOf: [{$ref: "#/$defs/bases"}]
    additionalProperties: {$ref: "#/$defs/text"}
  cost:
    description: Taken from the routed and compiled circuits before scheduling (docs/DESIGN.md, sections 5 and 9).
    type: object
    required: [swaps, router_swaps, qubits_used, depth, two_qubit_gates, two_qubit_gates_by_edge, pair_on_ports]
    additionalProperties: false
    properties:
      swaps: {type: integer, minimum: 0}
      router_swaps: {type: integer, minimum: 0}
      qubits_used: {type: array, items: {type: integer, minimum: 0}}
      depth: {type: integer, minimum: 0}
      two_qubit_gates: {type: integer, minimum: 0}
      two_qubit_gates_by_edge:
        allOf: [{$ref: "#/$defs/bases"}]
        additionalProperties:
          type: object
          additionalProperties: {type: integer, minimum: 0}
      pair_on_ports: {type: boolean}
  scheduled_duration:
    description: The scheduled ZZ circuit's duration in seconds; null for a profile without durations.
    oneOf: [{type: number, minimum: 0}, {type: "null"}]
  simulations:
    type: object
    required: [ideal, noisy]
    additionalProperties: false
    properties:
      ideal: {$ref: "#/$defs/simulation"}
      noisy: {$ref: "#/$defs/simulation"}
  provenance:
    type: object
    required: [commit, uncommitted, packages]
    additionalProperties: false
    properties:
      commit: {type: [string, "null"]}
      uncommitted:
        description: Whether src/ or config/ had uncommitted changes, or the run was made outside git.
        type: boolean
      packages:
        type: object
        additionalProperties: {type: string}

In [ ]:
%%repo_file config/bp.schema.runs.yaml
# config/bp.schema.runs.yaml
#
# Validates bp.data.runs.yaml in each benchmark folder,
# results/runs/benchmark/<name>/<stamp>/ or workspace/runs/benchmark/<name>/<stamp>/:
# the run list a benchmark writes, naming the benchmark file it was made from
# and the runs its comparison table was built from (docs/DESIGN.md, section 6).
# Each run is the folder beside the list named by its run id, and the folder's
# stamp is when the benchmark was made. src/bellbench/run.py checks a run list
# against this schema when it writes one.

$schema: "https://json-schema.org/draft/2020-12/schema"
title: Run list
description: The benchmark file a benchmark was made from, its run ids, and which run fills which case.

type: object
required: [benchmark, runs, cases]
additionalProperties: false

$defs:
  slug:
    type: string
    pattern: '^[a-z0-9][a-z0-9-]*$'

properties:
  benchmark:
    description: The benchmark file by its repo-relative path and the hash of its parsed content.
    type: object
    required: [path, hash]
    additionalProperties: false
    properties:
      path: {type: string, pattern: '^config/benchmark/bp\.data\.benchmark\.[a-z0-9][a-z0-9-]*\.yaml$'}
      hash: {type: string, pattern: '^sha256:[0-9a-f]{64}$'}
  runs:
    description: The benchmark file's run ids, in its order; each names the run folder beside this list.
    type: array
    minItems: 1
    uniqueItems: true
    items: {$ref: "#/$defs/slug"}
  cases:
    description: >-
      The benchmark file's cases, each one of the challenge's nine cases filled by a
      run and a simulation; empty when it has none.
    type: array
    items:
      type: object
      required: [case, run, simulation]
      additionalProperties: false
      properties:
        case: {type: integer, minimum: 1, maximum: 9}
        run: {$ref: "#/$defs/slug"}
        simulation: {enum: [ideal, noisy]}

<a id="s2"></a>

## 2. The task

**On each of three processor graphs, create the Bell pair |Φ+⟩ = (|00⟩ + |11⟩)/√2 between two fixed qubits, measure how close it comes, and test one strategy that protects it.** Two qubits share a unit vector in ℂ⁴, and |Φ+⟩ is the vector (1, 0, 0, 1)/√2. A two-qubit gate acts only between neighbours on the graph, so a pair made at one port must be moved, or routed, to the other, and each added gate adds noise.

We verify the pair by measuring the two ports in three bases and combining the three correlators into the fidelity:

$$F = \frac{1 + \langle XX\rangle - \langle YY\rangle + \langle ZZ\rangle}{4}$$

F = 1 is the target exactly; F above 1/2 shows entanglement. The challenge fixes nine cases, each graph under three conditions:

| Graph | Ideal | Noisy | Noisy, protected |
|---|---|---|---|
| 2016-inspired | Case 1 | Case 2 | Case 3 |
| Contemporary | Case 4 | Case 5 | Case 6 |
| Hyperbolic-inspired | Case 7 | Case 8 | Case 9 |

<details><summary><b>The linear algebra behind the formula</b></summary>

- A correlator ⟨BB⟩ is the expectation value of B ⊗ B in the pair's state ρ, Tr(ρ (B ⊗ B)), for B one of the Pauli matrices X = [[0, 1], [1, 0]], Y = [[0, −i], [i, 0]] and Z = [[1, 0], [0, −1]]. From shots it is P(00) + P(11) − P(01) − P(10).
- |Φ+⟩⟨Φ+| = (I⊗I + X⊗X − Y⊗Y + Z⊗Z)/4, so F = ⟨Φ+|ρ|Φ+⟩, the overlap of the prepared state with the target.
- ZZ sees a bit flip, XX a phase flip, and YY either; all three together tell |Φ+⟩ from the other three Bell states.
- A state that is not entangled cannot have F above 1/2, so 1/2 is the line a classical correlation cannot cross.

The full statement of the task, with its sources, is [CHALLENGE.md](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/CHALLENGE.md).

</details>

<a id="s3"></a>

## 3. Methods

<a id="s3-1"></a>

### 3.1 The three processor graphs

**We use the organisers' three coupling graphs exactly, with the ports they fix, and explain results by three features of each graph:** the hops between the ports, the paths between the ports that share no qubit (disjoint paths), and the most edges at any qubit. The three processor files come first; the figure and the table are drawn from them.

In [ ]:
%%repo_file config/processor/bp.data.processor.ibm2016.yaml
# config/processor/bp.data.processor.ibm2016.yaml
#
# The 2016-inspired reference processor model, the organisers' graph (docs/DESIGN.md, section 3).
# Validated by config/bp.schema.processor.yaml.

name: ibm2016
label: 2016-inspired reference
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 1 (the models cell), gives the T graph
  as an edge list with its ports; the organisers stated at the event that the
  graphs are exact. Ids and positions are the notebook's: slide 4's drawings carry
  no qubit numbers, so the notebook labels the qubits by position.
qubits:
  - {id: 0, position: [0, 2]}
  - {id: 1, position: [-1.55, 0.75]}
  - {id: 2, position: [1.55, 0.75]}
  - {id: 3, position: [0, -0.45]}
  - {id: 4, position: [0, -2]}
ports: [0, 4]
edges:
  - [0, 1]
  - [0, 2]
  - [0, 3]
  - [3, 4]

In [ ]:
%%repo_file config/processor/bp.data.processor.contemporary.yaml
# config/processor/bp.data.processor.contemporary.yaml
#
# The Contemporary IBM-style processor model, the organisers' graph (docs/DESIGN.md, section 3).
# Validated by config/bp.schema.processor.yaml.

name: contemporary
label: Contemporary IBM-style
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 1 (the models cell), gives the 9-node graph
  as an edge list with its ports; the organisers stated at the event that the
  graphs are exact. Ids and positions are the notebook's: slide 4's drawings carry
  no qubit numbers, so the notebook labels the qubits by position.
qubits:
  - {id: 0, position: [-2.1, 1]}
  - {id: 1, position: [-0.9, 1.7]}
  - {id: 2, position: [0.3, 1]}
  - {id: 3, position: [1.5, 1.7]}
  - {id: 4, position: [-1.5, -0.35]}
  - {id: 5, position: [-0.3, 0.35]}
  - {id: 6, position: [0.9, -0.35]}
  - {id: 7, position: [2.1, 0.35]}
  - {id: 8, position: [-0.3, -1.65]}
ports: [0, 7]
edges:
  - [0, 1]
  - [0, 4]
  - [1, 2]
  - [1, 5]
  - [2, 3]
  - [2, 5]
  - [2, 6]
  - [3, 7]
  - [4, 5]
  - [5, 6]
  - [5, 8]
  - [6, 7]

In [ ]:
%%repo_file config/processor/bp.data.processor.hyperbolic.yaml
# config/processor/bp.data.processor.hyperbolic.yaml
#
# The Hyperbolic-inspired processor model, the organisers' graph (docs/DESIGN.md, section 3).
# Validated by config/bp.schema.processor.yaml.

name: hyperbolic
label: Hyperbolic-inspired model
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 1 (the models cell), gives the 13-node graph
  as an edge list with its ports; the organisers stated at the event that the
  graphs are exact. Ids and positions are the notebook's: slide 4's drawings carry
  no qubit numbers, so the notebook labels the qubits by position.
qubits:
  - {id: 0, position: [0, 2.08]}
  - {id: 1, position: [-2, 1.14]}
  - {id: 2, position: [0, 1.14]}
  - {id: 3, position: [2, 1.14]}
  - {id: 4, position: [-1, 0.59]}
  - {id: 5, position: [1, 0.59]}
  - {id: 6, position: [0, 0]}
  - {id: 7, position: [-1, -0.59]}
  - {id: 8, position: [1, -0.59]}
  - {id: 9, position: [-2, -1.14]}
  - {id: 10, position: [0, -1.14]}
  - {id: 11, position: [2, -1.14]}
  - {id: 12, position: [0, -2.08]}
ports: [0, 12]
edges:
  - [0, 1]
  - [0, 3]
  - [1, 9]
  - [3, 11]
  - [9, 12]
  - [11, 12]
  - [2, 4]
  - [2, 5]
  - [4, 7]
  - [5, 8]
  - [7, 10]
  - [8, 10]
  - [0, 2]
  - [1, 4]
  - [3, 5]
  - [7, 9]
  - [8, 11]
  - [10, 12]
  - [2, 6]
  - [4, 6]
  - [5, 6]
  - [6, 7]
  - [6, 8]
  - [6, 10]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from bellbench.figures import processor_figure
from bellbench.paths import graph_metrics

fig = processor_figure()
display(fig)
plt.close(fig)
pd.DataFrame(
    {p: {k: graph_metrics(p)[k] for k in ("port_distance", "disjoint_paths", "max_degree", "path")}
     for p in configs.CORE_PROCESSORS}
).T.rename(columns={"port_distance": "hops between ports", "disjoint_paths": "disjoint paths",
                   "max_degree": "most edges at a qubit", "path": "pair's path"})

<a id="s3-2"></a>

### 3.2 The noise model

**Our noise model uses the calibration values of IBM's Boston processor, taken from the FakeBoston snapshot recorded on 2026-04-17, and reproduces Aer's own Boston noise channels exactly when given the same values.** Each value is the median over the snapshot's qubits and couplers, and every qubit and coupler of all three graphs takes it, so the graphs differ in connectivity alone. Nothing here ran on hardware.

- **Modelled:** gate errors, decay (T1 and T2) during gates and while idle, and readout errors that differ in the two directions. Circuits are scheduled before noise is applied, so a qubit waiting for a 2.2 µs measurement decays while it waits.
- **Set per run:** a mid-circuit measurement error, which case 3 needs because its check reads a qubit and reuses it; the move-chain benchmark sets it to 0.01.

In [ ]:
%%repo_file config/noise/bp.data.noise.boston-2026.yaml
# config/noise/bp.data.noise.boston-2026.yaml
#
# Medians of IBM's FakeBoston snapshot (Heron r3). Validated by
# config/bp.schema.noise.yaml; tests/test_config.py pins every value to the
# snapshot and every comment to the schema.

name: boston-2026
label: FakeBoston medians, calibrated 2026-04-17
kind: snapshot
reference: qiskit-fake-provider
snapshot:
  class: FakeBoston
  package: qiskit-ibm-runtime==0.51.0
  calibrated: "2026-04-17"
  properties: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/boston/props_boston.json
  configuration: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/boston/conf_boston.json
notes: >-
  Medians over the FakeBoston snapshot's 156 qubits and 352 directed CZ couplers, as
  qiskit-ibm-runtime 0.51.0 pins them. Every qubit and coupler of a processor
  graph takes these values.
# Time step in seconds: the hardware's clock tick. Null when the snapshot records none.
time_step: 4.0e-9
qubit:
  # T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1).
  t1_relaxation_time: 292.43565062547934e-6
  # T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1.
  t2_dephasing_time: 352.96959961159655e-6
  readout_error:
    # Readout error: the probability of reading 1 when the qubit was 0.
    reads_1_when_0: 0.00390625
    # Readout error: the probability of reading 0 when the qubit was 1.
    reads_0_when_1: 0.0048828125
# Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds.
gates:
  cz: {qubits: 2, error_probability: 0.0012747211439835399, duration: 68.0e-9}
  sx: {qubits: 1, error_probability: 0.00014470700669960387, duration: 32.0e-9}
  x: {qubits: 1, error_probability: 0.00014470700669960387, duration: 32.0e-9}
  rz: {qubits: 1, error_probability: 0.0, duration: 0.0}
# Measurement: its duration in seconds. Its errors are the readout errors.
measure: {duration: 2.2e-6}
# Reset to 0: its duration in seconds.
reset: {duration: 2.232e-6}
borrowed: []

In [ ]:
profile = configs.load_noise("boston-2026")
q, g = profile["qubit"], profile["gates"]
pd.DataFrame(
    [("T1", q["t1_relaxation_time"] * 1e6, "µs"), ("T2", q["t2_dephasing_time"] * 1e6, "µs"),
     ("reads 1 when 0", q["readout_error"]["reads_1_when_0"], ""), ("reads 0 when 1", q["readout_error"]["reads_0_when_1"], ""),
     ("CZ error", g["cz"]["error_probability"], ""), ("CZ duration", g["cz"]["duration"] * 1e9, "ns"),
     ("√X error", g["sx"]["error_probability"], ""), ("measurement", profile["measure"]["duration"] * 1e6, "µs")],
    columns=["quantity", "value", "unit"],
).set_index("quantity")

<details><summary><b>What the model leaves out</b></summary>

Measurement-induced dephasing, always-on ZZ coupling, spectator errors, coherent over- and under-rotations, drift, feedforward latency, cosmic-ray bursts, leakage, frequency crowding, and qubits that start a shot excited (appendix A.2 tests that one). Why each choice was made is [DESIGN.md, section 4](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/DESIGN.md#4-noise-model).

</details>

<a id="s3-3"></a>

### 3.3 The circuits

**Each run is one circuit file: an OpenQASM 3 circuit on the graph's physical qubits, and how its measured bits are read.** Three transports and one check fill the nine cases:

- **Baseline (cases 1, 2, 4, 5, 7, 8):** the organisers' recipe, H and CX at one port, SWAPs out to the other port and back, each SWAP three CX gates.
- **Move chain:** H and CX at the first port, then each hop moves the far half of the pair with two CX gates, a SWAP without its third CX, which is exact when the qubit it moves onto starts in |0⟩.
- **XX check (cases 3, 6, 9):** after the move chain, extra qubits measure the ports' X ⊗ X parity; a shot whose parity is odd is discarded. It catches a phase flip on one port and misses a bit flip. On the contemporary and hyperbolic-inspired graphs it runs on a spare path, sharing no qubit with the pair's, and reads only at the end; on the 2016-inspired graph it must reuse a qubit of the pair's path, so it reads that qubit mid-circuit first.

The 9 circuit files the move-chain benchmark runs, each written as the repository holds it: the circuit in OpenQASM 3, and the `analysis` that says how its check bits are read.

In [ ]:
%%repo_file config/circuit/bp.data.circuit.baseline-ibm2016.yaml
# config/circuit/bp.data.circuit.baseline-ibm2016.yaml

name: baseline-ibm2016
label: Baseline, 2016-inspired reference
processor: ibm2016
analysis:
  name: postselect
notes: >-
  The baseline transport (swap_chain in src/bellbench/circuits.py), the organisers'
  recipe: H on the first port, explicit SWAPs out along the path [0, 3, 4], the lexicographically
  smallest shortest path between the ports, CX onto the second port, and the same
  SWAPs back, so the pair ends on the ports. No checks, so postselection keeps every
  shot.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[5] q;
  h q[0];
  swap q[0], q[3];
  cx q[3], q[4];
  swap q[0], q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.baseline-contemporary.yaml
# config/circuit/bp.data.circuit.baseline-contemporary.yaml

name: baseline-contemporary
label: Baseline, Contemporary IBM-style
processor: contemporary
analysis:
  name: postselect
notes: >-
  The baseline transport (swap_chain in src/bellbench/circuits.py), the organisers'
  recipe: H on the first port, explicit SWAPs out along the path [0, 1, 2, 3, 7],
  the lexicographically smallest shortest path between the ports, CX onto the second
  port, and the same SWAPs back, so the pair ends on the ports. No checks, so postselection
  keeps every shot.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[0];
  swap q[0], q[1];
  swap q[1], q[2];
  swap q[2], q[3];
  cx q[3], q[7];
  swap q[2], q[3];
  swap q[1], q[2];
  swap q[0], q[1];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.baseline-hyperbolic.yaml
# config/circuit/bp.data.circuit.baseline-hyperbolic.yaml

name: baseline-hyperbolic
label: Baseline, Hyperbolic-inspired model
processor: hyperbolic
analysis:
  name: postselect
notes: >-
  The baseline transport (swap_chain in src/bellbench/circuits.py), the organisers'
  recipe: H on the first port, explicit SWAPs out along the path [0, 1, 9, 12], the
  lexicographically smallest shortest path between the ports, CX onto the second port,
  and the same SWAPs back, so the pair ends on the ports. No checks, so postselection
  keeps every shot.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[0];
  swap q[0], q[1];
  swap q[1], q[9];
  cx q[9], q[12];
  swap q[1], q[9];
  swap q[0], q[1];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-ibm2016.yaml
# config/circuit/bp.data.circuit.move-chain-ibm2016.yaml

name: move-chain-ibm2016
label: move_chain, ibm2016
processor: ibm2016
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: move_chain().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[4], q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-contemporary.yaml
# config/circuit/bp.data.circuit.move-chain-contemporary.yaml

name: move-chain-contemporary
label: move_chain, contemporary
processor: contemporary
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  move_chain().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-chain-hyperbolic.yaml

name: move-chain-hyperbolic
label: move_chain, hyperbolic
processor: hyperbolic
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: move_chain().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[1];
  cx q[9], q[12];
  cx q[12], q[9];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-path-xx-ibm2016.yaml
# config/circuit/bp.data.circuit.move-chain-path-xx-ibm2016.yaml

name: move-chain-path-xx-ibm2016
label: move_chain then bus_xx on the path, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks:
    - - check_vac_xx[0]
    - - check_xx[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: move_chain();
  then bus_xx(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_vac_xx;
  bit[1] check_xx;
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[4], q[3];
  check_vac_xx[0] = measure q[3];
  h q[3];
  cx q[3], q[0];
  cx q[3], q[4];
  h q[3];
  check_xx[0] = measure q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-spare-xx-contemporary.yaml
# config/circuit/bp.data.circuit.move-chain-spare-xx-contemporary.yaml

name: move-chain-spare-xx-contemporary
label: move_chain then bus_xx on a spare path, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  move_chain(); then bus_xx(on=[0, 4, 5, 6, 7], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_xx;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];
  h q[4];
  cx q[4], q[5];
  cx q[5], q[6];
  cx q[4], q[0];
  cx q[6], q[7];
  h q[4];
  check_xx[0] = measure q[4];
  h q[5];
  check_xx[1] = measure q[5];
  h q[6];
  check_xx[2] = measure q[6];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-spare-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-chain-spare-xx-hyperbolic.yaml

name: move-chain-spare-xx-hyperbolic
label: move_chain then bus_xx on a spare path, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: move_chain();
  then bus_xx(on=[0, 3, 11, 12], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_xx;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  h q[3];
  cx q[3], q[11];
  cx q[3], q[0];
  cx q[11], q[12];
  h q[3];
  check_xx[0] = measure q[3];
  h q[11];
  check_xx[1] = measure q[11];

<a id="s3-4"></a>

### 3.4 The benchmark files

**A benchmark file names its runs, each a circuit file under a noise profile with its settings, the shots and the base seed, and which run fills each case.** The core benchmark holds the organisers' baseline on each graph; the move-chain benchmark holds all nine cases, and this notebook reports it.

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.core.yaml
# config/benchmark/bp.data.benchmark.core.yaml

name: core
label: Core benchmark
notes: >-
  The challenge's nine cases (docs/CHALLENGE.md, The nine core cases): the three
  processor models under one noise profile, boston-2026, so the comparison shows
  connectivity alone. Each run is the organisers' baseline transport on its graph.
  Cases 3, 6 and 9, the protected runs, stay open until a protected strategy is
  chosen for the core; the move-chain benchmark fills them with one candidate.
  20,000 shots per basis gives F a standard error of about 0.001.
shots: 20000
base_seed: 20261116
runs:
  - {run: ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: contemporary, circuit: baseline-contemporary, noise: boston-2026}
  - {run: hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
cases:
  - {case: 1, run: ibm2016, simulation: ideal}
  - {case: 2, run: ibm2016, simulation: noisy}
  - {case: 4, run: contemporary, simulation: ideal}
  - {case: 5, run: contemporary, simulation: noisy}
  - {case: 7, run: hyperbolic, simulation: ideal}
  - {case: 8, run: hyperbolic, simulation: noisy}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.move-chain.yaml
# config/benchmark/bp.data.benchmark.move-chain.yaml

name: move-chain
label: Move chain with an XX check, the nine cases
notes: >-
  The challenge's nine cases (docs/CHALLENGE.md, The nine core cases) with one
  protection strategy filled in, under boston-2026 as in the core benchmark, so the
  three graphs differ only in connectivity. Cases 1, 2, 4, 5, 7 and 8 are the
  organisers' baseline, the reference, exactly as the core benchmark runs it. Cases
  3, 6 and 9 are the protection: move_chain carries the pair to the ports with two CX
  gates a hop instead of a SWAP's three, then bus_xx reads the ports' XX parity
  through a bus of extra qubits and postselection discards each shot where it reads
  -1. The check catches a phase flip on one port and is blind to a bit flip, to the
  same flip on both ports and to an error on the bus. On the contemporary and
  hyperbolic graphs the bus sits on a spare path that shares no qubit with the pair
  and reads only at the end; ibm2016 has no spare path, so there the bus sits on the
  pair's own path, reads each qubit as vacant before reusing it, and needs the
  mid-circuit measurement channel, set to 0.01 as one point of the range it is to be
  swept over. The move chain also runs without the check, filling no case, so the
  table separates what the cheaper transport buys from what the check buys. Every
  run takes 20,000 shots per basis, as in the core; protected runs are not yet given
  more shots in proportion to their acceptance.
shots: 20000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: baseline-contemporary, circuit: baseline-contemporary, noise: boston-2026}
  - {run: baseline-hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
  # The move chain alone, filling no case.
  - {run: move-ibm2016, circuit: move-chain-ibm2016, noise: boston-2026}
  - {run: move-contemporary, circuit: move-chain-contemporary, noise: boston-2026}
  - {run: move-hyperbolic, circuit: move-chain-hyperbolic, noise: boston-2026}
  # The protection: the move chain with its XX check.
  - {run: path-xx-ibm2016, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
cases:
  - {case: 1, run: baseline-ibm2016, simulation: ideal}
  - {case: 2, run: baseline-ibm2016, simulation: noisy}
  - {case: 3, run: path-xx-ibm2016, simulation: noisy}
  - {case: 4, run: baseline-contemporary, simulation: ideal}
  - {case: 5, run: baseline-contemporary, simulation: noisy}
  - {case: 6, run: spare-xx-contemporary, simulation: noisy}
  - {case: 7, run: baseline-hyperbolic, simulation: ideal}
  - {case: 8, run: baseline-hyperbolic, simulation: noisy}
  - {case: 9, run: spare-xx-hyperbolic, simulation: noisy}

**Every file the nine cases used is now on disk, so the session can start; the check below compares each case run's recorded hashes with the files above.** A run records the content hash of each file it read, so a match shows that these are the files behind the recorded numbers.

In [ ]:
import yaml
from bellbench.run import current_benchmark, read_table
from bellbench.workbench import Workbench

REPORTED = "move-chain"
wb = Workbench()
folder = current_benchmark(REPORTED)
checks = []
for run in yaml.safe_load((folder / "bp.data.runs.yaml").read_text())["runs"]:
    record = yaml.safe_load((folder / run / "bp.data.result.yaml").read_text())
    refs = [record["circuit"], record["noise_control"]["processor"], record["noise_control"]["noise"]]
    checks.append({"run": run, "files": len(refs),
                   "every hash matches": all(configs.content_hash(WORK / r["path"]) == r["hash"] for r in refs)})
pd.DataFrame(checks).set_index("run")

<a id="s3-5"></a>

### 3.5 The transpiled circuits

**Each run compiles its circuit once, at optimisation level 1, into Boston's native gates (CZ, √X, X and Rz); these are the transpiled circuits of the nine cases (challenge material 02), as their run records hold them.** For each graph, the workbench shows the baseline (the ideal and noisy cases) and the protected case: the graph with the couplers the circuit uses, the written circuit and its cost, then dropdowns: a timeline of when each qubit is busy in the compiled ZZ circuit (the XX and YY circuits add only their final rotations), a step-through of the written circuit with no noise (the baseline's shows the SWAPs out and back that the move chain avoids), the compiled ZZ circuit drawn, and every compiled circuit as OpenQASM 3. A step-through stops at a measurement made before the end of the circuit, so case 3's stops where its check reads a qubit mid-circuit.

In [ ]:
from html import escape

from qiskit import qasm3
from bellbench.views.base import emit, fold, img

CASES = {"ibm2016": ("baseline-ibm2016", "path-xx-ibm2016", "cases 1 and 2", "case 3"),
         "contemporary": ("baseline-contemporary", "spare-xx-contemporary", "cases 4 and 5", "case 6"),
         "hyperbolic": ("baseline-hyperbolic", "spare-xx-hyperbolic", "cases 7 and 8", "case 9")}


def transpiled(processor):
    """The workbench's view of a graph's baseline and protected runs, with their compiled circuits in dropdowns."""
    *runs, base_cases, protected_case = CASES[processor]
    for run, cases in zip(runs, (base_cases, protected_case)):
        result = wb.fetch(REPORTED, run)
        display(wb.show(result, step=True, timeline=True))
        fig = qasm3.loads(result.record["compiled"]["ZZ"]).draw("mpl", idle_wires=False, fold=40, scale=0.55)
        drawing = img(fig)
        plt.close(fig)
        emit([fold(f"{cases}: {run}, the compiled ZZ circuit, drawn", drawing)]
             + [fold(f"{cases}: {run}, the compiled {basis} circuit as OpenQASM 3", f"<pre>{escape(text)}</pre>")
                for basis, text in result.record["compiled"].items()])

#### The 2016-inspired graph: cases 1 to 3

In [ ]:
transpiled("ibm2016")

#### The contemporary graph: cases 4 to 6

In [ ]:
transpiled("contemporary")

#### The hyperbolic-inspired graph: cases 7 to 9

In [ ]:
transpiled("hyperbolic")

<a id="s3-6"></a>

### 3.6 What each run reports

**Each case reports F with its standard error, the acceptance (the fraction of shots kept), and the circuit's cost,** as the challenge's five measures require. Each run takes 20,000 shots per basis, simulated ideal and noisy with seeds derived from one base seed, 20261116, so every run repeats exactly.

- **F after postselection** is F of the kept shots; **F raw** is F of every shot. A protected result is never shown without its acceptance.
- **Standard error:** a correlator c from N shots has variance (1 − c²)/N; the three bases are separate shots, so SE(F) = √(Σ var)/4, with N the kept shots.
- **Cost:** two-qubit gates, depth, SWAPs and qubits used, from the compiled circuit before scheduling, and the scheduled duration.
- **Two-qubit gates per accepted pair:** the gates divided by the acceptance, what each kept pair costs.
- **Infidelity removed per rejection:** (F − F raw)/(1 − p), how much a check lowers 1 − F for each unit of shots it discards.

<a id="s3-7"></a>

### 3.7 Why we wrote the transports, rather than leave routing to the transpiler

**Qiskit's transpiler solves a different problem: it routes a circuit and tracks where its qubits end up, rather than bringing the pair back to the ports.** The cell gives the bare task, H on the first port then CX between the ports, to `generate_preset_pass_manager` at levels 1 to 3 with the graph's coupling map, CX as the two-qubit gate and the identity layout, over 20 seeds each, and counts the CX gates and how often the pair ends on the ports. [check]

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

rows = []
for processor in configs.CORE_PROCESSORS:
    proc = configs.load_processor(processor)
    edges = list(configs.processor_graph(proc).edges)
    coupling = CouplingMap([list(e) for e in edges] + [[b, a] for a, b in edges])
    a, b = proc["ports"]
    task = QuantumCircuit(len(proc["qubits"]))
    task.h(a)
    task.cx(a, b)
    for level in (1, 2, 3):
        cx, on_ports = set(), 0
        for seed in range(20):
            pm = generate_preset_pass_manager(level, coupling_map=coupling, basis_gates=["cx", "h", "rz", "sx", "x"],
                                              initial_layout=list(range(task.num_qubits)), seed_transpiler=seed)
            out = pm.run(task)
            cx.add(out.count_ops().get("cx", 0))
            final = out.layout.final_index_layout()
            on_ports += final[a] == a and final[b] == b
        rows.append({"graph": processor, "level": level, "CX gates": sorted(cx), "pair on the ports": f"{on_ports} of 20"})
moves = {p: wb.fetch(REPORTED, f"move-{p}").record["cost"]["two_qubit_gates"] for p in configs.CORE_PROCESSORS}
display(pd.DataFrame(rows).set_index(["graph", "level"]))
print("move chain, pair on the ports every time:", moves)

<a id="s4"></a>

## 4. Results: the nine cases

<a id="s4-1"></a>

### 4.1 The nine cases

**Every ideal case gives F = 1 with every shot kept, and the protected cases keep 97.0% to 98.3% of their shots.** One noise profile, one base seed, 20,000 shots per basis; each ± is one standard error.

| Graph | Ideal | Noisy baseline | Protected: F after postselection, acceptance |
|---|---|---|---|
| 2016-inspired | 1 (case 1) | 0.9770 ± 0.0008 (case 2) | 0.9786 ± 0.0007, 0.970 (case 3) |
| Contemporary | 1 (case 4) | 0.9614 ± 0.0010 (case 5) | 0.9793 ± 0.0007, 0.975 (case 6) |
| Hyperbolic-inspired | 1 (case 7) | 0.9681 ± 0.0009 (case 8) | 0.9812 ± 0.0007, 0.983 (case 9) |

The table below gives every case the five measures the challenge asks every result to report, from the run records of the move-chain benchmark, the benchmark that fills the nine cases.

In [ ]:
import numpy as np

BASES = ("XX", "YY", "ZZ")
cases = read_table(current_benchmark(REPORTED) / "comparison.csv").dropna(subset=["case"]).sort_values("case")
raw = sum(cases[f"shots_raw_{b}"] for b in BASES)
kept = sum(cases[f"shots_accepted_{b}"] for b in BASES)
CONDITION = {1: "ideal", 2: "noisy baseline", 0: "noisy, protected"}
table = pd.DataFrame({
    "case": cases.case.astype(int),
    "graph": cases.processor,
    "condition": [CONDITION[int(c) % 3] for c in cases.case],
    "F ± SE": [f"{f:.4f} ± {e:.4f}" for f, e in zip(cases.fidelity, cases.fidelity_se)],
    "F raw": cases.fidelity_raw.round(4),
    "acceptance (95% interval)": [f"{a:.3f} ({lo:.3f} to {hi:.3f})"
                                  for a, lo, hi in zip(cases.acceptance, cases.acceptance_low, cases.acceptance_high)],
    "discarded shots": (raw - kept).astype(int),
    "two-qubit gates": cases.two_qubit_gates,
    "depth": cases.depth,
    "SWAPs": cases.swaps,
    "qubits used": cases.qubits_used,
    "schedule (µs)": (cases.scheduled_duration * 1e6).round(2),
}).set_index("case")
display(table)

noisy = cases[cases.simulation == "noisy"]
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4), sharey=True)
for ax, processor in zip(axes, configs.CORE_PROCESSORS):
    rows = noisy[noisy.processor == processor]
    labels = [f"case {int(c)}\n{'baseline' if int(c) % 3 == 2 else 'protected'}" for c in rows.case]
    colours = ["#9aa5b8" if int(c) % 3 == 2 else "#7139cc" for c in rows.case]
    ax.bar(labels, rows.fidelity, yerr=rows.fidelity_se, color=colours, capsize=4)
    for x, (f, a) in enumerate(zip(rows.fidelity, rows.acceptance)):
        ax.annotate(f"{f:.4f}" + ("" if a == 1 else f"\nkept {a:.3f}"), (x, f), textcoords="offset points",
                    xytext=(0, 6), ha="center", fontsize=8)
    ax.set_title(processor)
low = float(np.floor((noisy.fidelity - noisy.fidelity_se).min() / 0.005) * 0.005) - 0.005
high = float((noisy.fidelity + noisy.fidelity_se).max()) + 0.007  # room for the labels above the bars
axes[0].set_ylim(low, high)
axes[0].set_ylabel(f"F, noisy (± 1 SE)\naxis starts at {low:.3f}, not 0")
fig.suptitle("The six noisy cases; the ideal cases 1, 4 and 7 give F = 1 exactly", fontsize=10)
fig.tight_layout()
display(fig)
plt.close(fig)

<a id="s4-2"></a>

### 4.2 What protection buys, and what it costs

**Most of the gain over the baseline comes from the transport, and the check adds to it only where it reads a spare path at the end of the circuit.** For each graph, the workbench sets the baseline, the move chain alone and the move chain with its XX check side by side: a table, then a bar per run of its F, and a second bar for a run's kept shots, with its acceptance. Their circuits are in section 3.5.

| Graph | Baseline | Move chain | Move chain + XX check | The check's cost |
|---|---|---|---|---|
| 2016-inspired | 0.9770 ± 0.0008, 7 gates | 0.9805 ± 0.0007, 3 gates | 0.9786 ± 0.0007 at 0.970, 5 gates | a 2.2 µs mid-circuit read; schedule 2.50 → 4.93 µs |
| Contemporary | 0.9614 ± 0.0010, 19 gates | 0.9756 ± 0.0008, 7 gates | 0.9793 ± 0.0007 at 0.975, 11 gates | 4 more gates, 3 more qubits |
| Hyperbolic-inspired | 0.9681 ± 0.0009, 13 gates | 0.9788 ± 0.0007, 5 gates | 0.9812 ± 0.0007 at 0.983, 8 gates | 3 more gates, 2 more qubits |

In [ ]:
def side_by_side(processor):
    """The baseline, the move chain and the protected run on one graph, as the workbench compares them."""
    baseline, protected, *_ = CASES[processor]
    return wb.compare(*(wb.fetch(REPORTED, run) for run in (baseline, f"move-{processor}", protected)))

#### The 2016-inspired graph: the check does not pay

**Its mid-circuit read holds the pair idle for 2.2 µs, and F after postselection, 0.9786 ± 0.0007, stays below the move chain alone, 0.9805 ± 0.0007.** The graph has one path between its ports, so the check must reuse a qubit of the pair's path and read it mid-circuit first; appendix A.1 shows this holds even with no mid-circuit measurement error. [check]

In [ ]:
side_by_side("ibm2016")

#### The contemporary graph: the check pays

**The check, on a spare path read at the end, adds 0.0037 to the move chain's F, more than twice the combined standard error, keeping 97.5% of shots.** The move chain alone already recovers most of the baseline's loss, from 19 two-qubit gates to 7. [check]

In [ ]:
side_by_side("contemporary")

#### The hyperbolic-inspired graph: the check pays

**The check adds 0.0024 to the move chain's F, more than twice the combined standard error, keeping 98.3% of shots, with 3 more two-qubit gates and 2 more qubits.** [check]

In [ ]:
side_by_side("hyperbolic")

**Across the three graphs,** F of the kept shots against what each kept pair costs in two-qubit gates:

In [ ]:
noisy = read_table(current_benchmark(REPORTED) / "comparison.csv").query("simulation == 'noisy'")
COLOURS = dict(zip(configs.CORE_PROCESSORS, ("#2151c7", "#137665", "#c62639")))
MARKERS = {"baseline": "o", "move": "s", "path": "^", "spare": "^"}
fig, ax = plt.subplots(figsize=(7, 4))
for _, r in noisy.iterrows():
    ax.errorbar(r.two_qubit_gates_per_accepted_pair, r.fidelity, yerr=r.fidelity_se, fmt=MARKERS[r.run.split("-")[0]],
                color=COLOURS[r.processor], capsize=3, markersize=7)
handles = [plt.Line2D([], [], color=c, marker="o", ls="", label=p) for p, c in COLOURS.items()]
handles += [plt.Line2D([], [], color="0.4", marker=m, ls="", label=l)
            for m, l in (("o", "baseline"), ("s", "move chain"), ("^", "move chain + XX check"))]
ax.legend(handles=handles, fontsize=8, ncol=2)
ax.set(xlabel="two-qubit gates per accepted pair", ylabel="F of the kept shots (± 1 SE)",
       title=f"Quality against cost, noisy runs of the {REPORTED} benchmark")
display(fig)
plt.close(fig)

<a id="s4-3"></a>

### 4.3 What the graphs explain

**All three graphs share one noise model, so their differences come from connectivity alone. Port distance sets the cost of moving the pair, and the number of disjoint paths sets whether a check can read at the end.** [check]

| Graph | Hops between ports | Disjoint paths | Baseline gates, F | Move chain gates, F | Protected F, acceptance |
|---|---|---|---|---|---|
| 2016-inspired | 2 | 1 | 7, 0.9770 | 3, 0.9805 | 0.9786, 0.970 |
| Hyperbolic-inspired | 3 | 3 | 13, 0.9681 | 5, 0.9788 | 0.9812, 0.983 |
| Contemporary | 4 | 2 | 19, 0.9614 | 7, 0.9756 | 0.9793, 0.975 |

- **Unprotected, F falls as the ports grow apart:** the baseline and the move chain both rank the graphs by hops, 2, then 3, then 4.
- **Protected, the ranking changes:** the hyperbolic-inspired graph, with three disjoint paths, gives the highest F; the 2016-inspired graph, with one, falls from first to last, though its protected F and the contemporary graph's differ by less than one combined standard error.
- **The hyperbolic-inspired graph is a schematic,** a 13-qubit graph of short distances and high degree, not a patch of a hyperbolic lattice; we claim no advantage from its geometry beyond these three features.

<a id="s4-4"></a>

### 4.4 Rerun a case

**Any run can be made again from its record; at the recorded shots and seeds it repeats every number exactly.** The cell reruns case 3, compiling and simulating it again, and compares every circuit, count and number with its record. Outside the team's git repository the commit cannot match, and on another machine the package versions may differ; the numbers are what this checks.

In [ ]:
from bellbench.run import rerun

code, lines = rerun(current_benchmark(REPORTED) / CASES["ibm2016"][1])
print("\n".join(lines))

<a id="s5"></a>

## 5. Interpretation

**What the results show, against the three things the challenge asks a comparison to explain:** [check]

- **Fidelity against the operations compilation added.** Under one noise model, F tracks the two-qubit gates and the idle time the circuit adds. The baseline's SWAPs out and back cost 7, 13 and 19 two-qubit gates; the move chain's 3, 5 and 7 recover most of the lost fidelity on every graph.
- **Whether protection pays.** A check pays when it reads at the end of the circuit, on qubits the pair never used: on the contemporary and hyperbolic-inspired graphs it raises F by more than twice its standard error while discarding 1.7% to 2.5% of shots. A check that must read mid-circuit, on the 2016-inspired graph, holds the pair idle while it reads and does not raise F beyond its standard error at any measurement error we set.
- **Connectivity against noise.** Every graph runs under the same profile, so the differences are connectivity's: hops between the ports set the unprotected cost, and disjoint paths decide whether a check can avoid a mid-circuit read.

<a id="s6"></a>

## 6. Limitations

**These results describe circuits under one assumed noise model, simulated; they are not a measurement of any device.** [check]

- **One noise model:** Boston's calibration medians. Its readout pair puts every absolute F about 0.002 high, equally on all three graphs, so comparisons between graphs stand. What it leaves out, and why, is in [DESIGN.md, section 4](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/DESIGN.md#4-noise-model).
- **How far the results hold:** four optional studies each vary one assumption under the same profile ([appendix A](#sA)). case 3's check does not beat the move chain alone even with no mid-circuit measurement error; the move chain's lead over the baseline on the contemporary graph is gone by a starting excitation of 0.005; moving the pair out from the middle gives no gain we claim; and a check that reads the emptied path at the end beats case 3 on every measure.
- **One seed per run:** a difference smaller than about 0.001 needs more seeds before it means anything.
- **Detection, not correction:** a check discards shots and corrects nothing, and no fault injector has tested the checks fault by fault.
- **Idle qubits start in |0⟩:** the move chain and the spare-path checks rely on it; appendix A.2 shows how fast the lead shrinks when they do not.
- **No degree penalty:** case 6's check passes through the contemporary graph's five-edge qubit with no extra error.
- **The graphs:** the contemporary graph is the organisers' 9-node graph, not a real IBM layout, and the hyperbolic-inspired graph is a schematic.

<a id="s7"></a>

## 7. Recommendation

**Give a future cloud processor at least two qubit-disjoint paths between qubits that must share entanglement, so a parity check can run on a spare path and read at the end of the circuit.** [check]

On the two graphs with spare paths, the XX check raised F above the move chain alone (section 4.2); on the graph with one path, the check had to read a qubit mid-circuit and did not pay, even with no mid-circuit measurement error (appendix A.1). The vacancy-read study points the same way: a check that reads at the end, even on the pair's own path, beat case 3 on every measure (appendix A.4). The recommendation is conditional on our Boston-based model, and qubits that start a shot excited weaken it (appendix A.2).

<a id="sA"></a>

## Appendix A. Optional studies

**Four studies each vary one assumption behind the nine cases; none fills a case.** All four run under the Boston profile, and each point rests on one seed, so a difference smaller than about 0.001 needs more seeds before it means anything. Each study's benchmark and circuit files are in [appendix B](#sB).

In [ ]:
def study_table(name):
    """Every noisy run of a study, its numbers, in a dropdown."""
    t = read_table(current_benchmark(name) / "comparison.csv").query("simulation == 'noisy'")
    cols = ["run", "fidelity", "fidelity_se", "fidelity_raw", "acceptance", "two_qubit_gates", "scheduled_duration"]
    emit([fold(f"Every run of {name}, noisy simulations", t[cols].round(4).to_html(index=False))])


def sweep(name, label, groups):
    """F of the kept shots against a swept value, one line per circuit."""
    t = read_table(current_benchmark(name) / "comparison.csv").query("simulation == 'noisy'").copy()
    t["value"] = t.run.str.extract(r"-(?:exc|mcm)-([0-9e-]+)$")[0].astype(float)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for prefix, colour in groups:
        rows = t[t.run.str.startswith(prefix + "-")].sort_values("value")
        ax.errorbar(rows.value, rows.fidelity, yerr=rows.fidelity_se, fmt="o-", capsize=3, color=colour, label=prefix)
    ax.set(xlabel=label, ylabel="F of the kept shots (± 1 SE)")
    ax.legend(fontsize=8, ncol=2)
    display(fig)
    plt.close(fig)

<a id="sA-1"></a>

### A.1 Mid-circuit measurement error

**Case 3's check never beats the move chain alone by more than its standard error, at any mid-circuit measurement error we set, 0 included: the 2.2 µs wait, not the error, is its cost.** At 0, case 3 gives 0.9796 ± 0.0007 keeping 97.9%, against the move chain's 0.9805 ± 0.0007; higher errors discard more shots, 6.6% at 0.05, without lowering F. [check]

In [ ]:
sweep("mid-circuit-sweep", "mid-circuit measurement error", [("path-xx-ibm2016-mcm", "#e07b00")])
study_table("mid-circuit-sweep")

<a id="sA-2"></a>

### A.2 Starting excitation

**The move chain and the spare-path checks assume idle qubits start in |0⟩, so their lead shrinks fastest when a qubit may start a shot in |1⟩; our results hold only while that chance stays well below 0.005.** At 0.02 the move chain on the contemporary graph falls to 0.8839 ± 0.0016, below the baseline's 0.9238 ± 0.0013. Case 3's check, which reads each qubit as vacant before reusing it, loses least. [check]

In [ ]:
sweep("starting-excitation", "starting excitation",
      [("baseline-contemporary", "#9aa5b8"), ("move-contemporary", "#137665"), ("spare-xx-contemporary", "#7139cc"),
       ("baseline-ibm2016", "#c9b0a0"), ("move-ibm2016", "#c62639"), ("path-xx-ibm2016", "#e07b00")])
study_table("starting-excitation")

<a id="sA-3"></a>

### A.3 Middle out

**Making the pair mid-path and moving both halves out at once gave no gain we claim.** With the check, F moved by less than its standard error on both graphs. Without it, the contemporary graph's F rose from 0.9756 ± 0.0008 to 0.9790 ± 0.0007, a difference that rests on one seed and does not appear with the check on. A ZZ check before the XX check on the hyperbolic-inspired graph's third path cost acceptance, 0.963 against 0.983, without raising F. [check]

In [ ]:
wb.benchmark("middle-out")

<a id="sA-4"></a>

### A.4 Vacancy read

**A transport that empties the path with gates from the first port, then reads the emptied qubits at the end, beat case 3 on every measure on the 2016-inspired graph: 0.9844 ± 0.0006 keeping 99.3%, with 3 two-qubit gates, a 2.53 µs schedule and no mid-circuit read.** On the other two graphs, followed by the XX check, it raised F at a cost in acceptance. The nine cases still report the move chain with its XX check. [check]

In [ ]:
wb.benchmark("vacancy-read")

<a id="sB"></a>

## Appendix B. The studies' files

**Each study's benchmark file, and the circuits it adds beyond the nine cases' own,** written as the repository holds them. The tables in appendix A come from the runs these files define.

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.mid-circuit-sweep.yaml
# config/benchmark/bp.data.benchmark.mid-circuit-sweep.yaml

name: mid-circuit-sweep
label: Mid-circuit measurement error sweep on ibm2016
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): how the XX check on ibm2016 fares as its mid-circuit measurement
  error grows. ibm2016 has no spare path, so its check sits on the pair's own path,
  reads each bus qubit mid-circuit and reuses it, and the move-chain benchmark's case 3
  sets the mid-circuit measurement error to 0.01, one assumed value. Here that circuit,
  move_chain with bus_xx on the pair's path, runs under boston-2026 at mid-circuit
  measurement errors 0, 0.001, 0.003, 0.01, 0.02 and 0.05, each run named by its value
  in e notation after mcm. Each value is one point of a sweep, not a claimed property
  of any device, and each run derives its own seeds, so neighbouring points also differ
  by shot noise: read each difference against its standard errors. The run at 0.01 has
  case 3's settings and seeds, so it gives case 3's numbers. The organisers' baseline
  and move_chain alone read nothing
  mid-circuit, so the error cannot reach them: each runs once, without the channel, as
  a fixed reference. Setting the check's F after postselection, with its acceptance,
  beside move_chain alone's F at each point shows at which errors, if any, the check
  pays for the shots it discards. 20,000 shots per basis and base seed 20261116, as in
  the core.
shots: 20000
base_seed: 20261116
runs:
  # The fixed references, as the move-chain benchmark runs them.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: move-ibm2016, circuit: move-chain-ibm2016, noise: boston-2026}
  # The move chain with its XX check on the pair's path, one run per mid-circuit measurement error.
  - {run: path-xx-ibm2016-mcm-0, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.0}}
  - {run: path-xx-ibm2016-mcm-1e-3, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.001}}
  - {run: path-xx-ibm2016-mcm-3e-3, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.003}}
  - {run: path-xx-ibm2016-mcm-1e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: path-xx-ibm2016-mcm-2e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.02}}
  - {run: path-xx-ibm2016-mcm-5e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.05}}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.starting-excitation.yaml
# config/benchmark/bp.data.benchmark.starting-excitation.yaml

name: starting-excitation
label: Starting excitation sweep on the move-chain benchmark's runs
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): how the move-chain benchmark's runs fare if each qubit may start
  a shot in |1> rather than |0>. The move chain's two-CX move is a SWAP without its
  third CX, so it assumes the qubit it moves onto starts in |0>, and the XX check on a
  spare path trusts its spare qubits to start in |0>; the organisers' baseline swaps
  the pair out and back, which returns an excited qubit on its path to where it
  started. Each of the move-chain benchmark's nine runs, the baseline, the move chain
  alone and the move chain with its XX check on each graph, runs under boston-2026 with
  the starting excitation, the chance that each qubit starts a shot in |1>, at 0,
  0.005, 0.01 and 0.02, each run named by its value in e notation after exc; the check
  on ibm2016 keeps case 3's mid-circuit measurement error of 0.01. A run at 0 differs
  from the move-chain benchmark's run of the same circuit only in its seeds. Each value is one
  point of a sweep, not a claimed property of any device, and each run derives its own
  seeds, so neighbouring points also differ by shot noise: read each difference
  against its standard errors. 20,000 shots per basis and base seed 20261116, as in
  the core.
shots: 20000
base_seed: 20261116
runs:
  # The organisers' baseline.
  - {run: baseline-ibm2016-exc-0, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: baseline-ibm2016-exc-5e-3, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: baseline-ibm2016-exc-1e-2, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: baseline-ibm2016-exc-2e-2, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: baseline-contemporary-exc-0, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: baseline-contemporary-exc-5e-3, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: baseline-contemporary-exc-1e-2, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: baseline-contemporary-exc-2e-2, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: baseline-hyperbolic-exc-0, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: baseline-hyperbolic-exc-5e-3, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: baseline-hyperbolic-exc-1e-2, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: baseline-hyperbolic-exc-2e-2, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  # The move chain alone.
  - {run: move-ibm2016-exc-0, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: move-ibm2016-exc-5e-3, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: move-ibm2016-exc-1e-2, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: move-ibm2016-exc-2e-2, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: move-contemporary-exc-0, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: move-contemporary-exc-5e-3, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: move-contemporary-exc-1e-2, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: move-contemporary-exc-2e-2, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: move-hyperbolic-exc-0, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: move-hyperbolic-exc-5e-3, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: move-hyperbolic-exc-1e-2, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: move-hyperbolic-exc-2e-2, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  # The move chain with its XX check, on ibm2016 with case 3's mid-circuit measurement error.
  - {run: path-xx-ibm2016-exc-0, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.0}}
  - {run: path-xx-ibm2016-exc-5e-3, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.005}}
  - {run: path-xx-ibm2016-exc-1e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.01}}
  - {run: path-xx-ibm2016-exc-2e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.02}}
  - {run: spare-xx-contemporary-exc-0, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: spare-xx-contemporary-exc-5e-3, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: spare-xx-contemporary-exc-1e-2, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: spare-xx-contemporary-exc-2e-2, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: spare-xx-hyperbolic-exc-0, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: spare-xx-hyperbolic-exc-5e-3, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: spare-xx-hyperbolic-exc-1e-2, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: spare-xx-hyperbolic-exc-2e-2, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.02}}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.middle-out.yaml
# config/benchmark/bp.data.benchmark.middle-out.yaml

name: middle-out
label: Middle-out study
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): do two variants of the reported strategy do better than it? The
  first, move_middle_out, makes the pair in the middle of the path and moves both
  halves outward at once, so it uses the same gates as move_chain in a shorter
  schedule. The second adds a ZZ check before the XX check on the hyperbolic graph,
  whose third qubit-disjoint path between the ports holds a second spare bus; the ZZ
  check catches a bit flip on one port, which the XX check misses, and runs first so
  that the XX check's parity covers the ZZ check's frame bit. The contemporary graph
  has only two qubit-disjoint paths, so it has no room for a second spare bus, and on
  ibm2016 move_middle_out writes the same circuit as move_chain, so neither graph runs
  that variant. Each new run sits beside the move-chain benchmark's run it varies,
  rerun here under the same circuit, profile and base seed, so with the same seeds and
  the same numbers. The ZZ and XX run's XX bus passes through the centre qubit, so its
  numbers are without the planned degree penalty (docs/DESIGN.md, section 4). Every
  run takes 20,000 shots per basis, as in the core.
shots: 20000
base_seed: 20261116
runs:
  # The reference: the move-chain benchmark's runs these vary.
  - {run: move-contemporary, circuit: move-chain-contemporary, noise: boston-2026}
  - {run: move-hyperbolic, circuit: move-chain-hyperbolic, noise: boston-2026}
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
  # Middle-out, without and with the XX check.
  - {run: middle-out-contemporary, circuit: move-middle-out-contemporary, noise: boston-2026}
  - {run: middle-out-hyperbolic, circuit: move-middle-out-hyperbolic, noise: boston-2026}
  - {run: middle-out-xx-contemporary, circuit: move-middle-out-spare-xx-contemporary, noise: boston-2026}
  - {run: middle-out-xx-hyperbolic, circuit: move-middle-out-spare-xx-hyperbolic, noise: boston-2026}
  # Middle-out with a ZZ check, then the XX check, on the hyperbolic graph's two spare paths.
  - {run: middle-out-zz-xx-hyperbolic, circuit: move-middle-out-spare-zz-xx-hyperbolic, noise: boston-2026}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.vacancy-read.yaml
# config/benchmark/bp.data.benchmark.vacancy-read.yaml

name: vacancy-read
label: Vacancy-read study
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): does a GHZ-and-uncompute transport, followed by a vacancy read
  of the emptied path at the end, do better than the reported strategy? ghz_uncompute
  makes a GHZ state along the path, then empties the interior with gates controlled
  from the first port, so the vacancy read at the end catches a bit flip on that port,
  the qubit that has waited longest. On ibm2016 the vacancy read replaces case 3's XX
  check on the path, which reads a qubit mid-circuit and reuses it; on the
  contemporary and hyperbolic graphs the reported XX check on a spare path follows it.
  Each new run sits beside the move-chain benchmark's protected run on its graph,
  rerun here under the same circuit, profile and base seed, so with the same seeds and
  the same numbers. Case 3's run appears twice: at the mid-circuit measurement error
  of 0.01 it is reported with, and at 0, the Boston profile alone, under which every
  new run here needs no channel. The contemporary runs' XX bus passes through the
  five-edge qubit, so their numbers are without the planned degree penalty
  (docs/DESIGN.md, section 4). Every run takes 20,000 shots per basis, as in the core.
shots: 20000
base_seed: 20261116
runs:
  # The reference: the move-chain benchmark's protected runs, case 3's also without the mid-circuit error.
  - {run: path-xx-ibm2016, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: path-xx-ibm2016-mcm-0, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.0}}
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
  # GHZ and uncompute, then a vacancy read at the end; then the XX check on a spare path where one exists.
  - {run: ghz-vacancy-ibm2016, circuit: ghz-uncompute-vacancy-ibm2016, noise: boston-2026}
  - {run: ghz-vacancy-xx-contemporary, circuit: ghz-uncompute-vacancy-spare-xx-contemporary, noise: boston-2026}
  - {run: ghz-vacancy-xx-hyperbolic, circuit: ghz-uncompute-vacancy-spare-xx-hyperbolic, noise: boston-2026}

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-contemporary.yaml
# config/circuit/bp.data.circuit.move-middle-out-contemporary.yaml

name: move-middle-out-contemporary
label: move_middle_out, contemporary
processor: contemporary
analysis:
  name: postselect
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[1];
  cx q[1], q[2];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-middle-out-hyperbolic.yaml

name: move-middle-out-hyperbolic
label: move_middle_out, hyperbolic
processor: hyperbolic
analysis:
  name: postselect
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[9], q[12];
  cx q[12], q[9];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-spare-xx-contemporary.yaml
# config/circuit/bp.data.circuit.move-middle-out-spare-xx-contemporary.yaml

name: move-middle-out-spare-xx-contemporary
label: move_middle_out then bus_xx on a spare path, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_xx;
  qubit[9] q;
  h q[1];
  cx q[1], q[2];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];
  h q[4];
  cx q[4], q[5];
  cx q[5], q[6];
  cx q[4], q[0];
  cx q[6], q[7];
  h q[4];
  check_xx[0] = measure q[4];
  h q[5];
  check_xx[1] = measure q[5];
  h q[6];
  check_xx[2] = measure q[6];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-spare-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-middle-out-spare-xx-hyperbolic.yaml

name: move-middle-out-spare-xx-hyperbolic
label: move_middle_out then bus_xx on a spare path, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_xx;
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  h q[3];
  cx q[3], q[11];
  cx q[3], q[0];
  cx q[11], q[12];
  h q[3];
  check_xx[0] = measure q[3];
  h q[11];
  check_xx[1] = measure q[11];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-spare-zz-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-middle-out-spare-zz-xx-hyperbolic.yaml

name: move-middle-out-spare-zz-xx-hyperbolic
label: move_middle_out then bus_zz and bus_xx on two spare paths, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_zz[0]
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
      - frame_z_zz[0]
    frame_z:
    - frame_z_zz[0]
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_zz;
  bit[1] frame_z_zz;
  bit[3] check_xx;
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  cx q[0], q[3];
  cx q[3], q[11];
  cx q[12], q[11];
  check_zz[0] = measure q[11];
  h q[3];
  frame_z_zz[0] = measure q[3];
  h q[2];
  cx q[2], q[6];
  cx q[6], q[10];
  cx q[2], q[0];
  cx q[10], q[12];
  h q[2];
  check_xx[0] = measure q[2];
  h q[6];
  check_xx[1] = measure q[6];
  h q[10];
  check_xx[2] = measure q[10];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-ibm2016.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-ibm2016.yaml

name: ghz-uncompute-vacancy-ibm2016
label: ghz_uncompute then vacancy at the end, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: ghz_uncompute();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_vac;
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[0], q[3];
  check_vac[0] = measure q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-contemporary.yaml

name: ghz-uncompute-vacancy-spare-xx-contemporary
label: ghz_uncompute then vacancy, then bus_xx on a spare path, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_vac[2]
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_uncompute(); then vacancy(); then bus_xx(on=[0, 4, 5, 6, 7], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_vac;
  bit[3] check_xx;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[3];
  cx q[1], q[2];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[2];
  check_vac[2] = measure q[3];
  h q[4];
  cx q[4], q[5];
  cx q[5], q[6];
  cx q[4], q[0];
  cx q[6], q[7];
  h q[4];
  check_xx[0] = measure q[4];
  h q[5];
  check_xx[1] = measure q[5];
  h q[6];
  check_xx[2] = measure q[6];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-hyperbolic.yaml

name: ghz-uncompute-vacancy-spare-xx-hyperbolic
label: ghz_uncompute then vacancy, then bus_xx on a spare path, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_xx[0]
      - check_xx[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_uncompute();
  then vacancy(); then bus_xx(on=[0, 3, 11, 12], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_vac;
  bit[2] check_xx;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[9];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[9];
  h q[3];
  cx q[3], q[11];
  cx q[3], q[0];
  cx q[11], q[12];
  h q[3];
  check_xx[0] = measure q[3];
  h q[11];
  check_xx[1] = measure q[11];

<a id="sC"></a>

## Appendix C. Config files beyond the Boston runs

**Every config file nothing above writes: the other noise profiles, which only the optional biased-noise study and other cross-checks use, and the benchmarks this notebook does not report.** They complete the repository's config, so every assumption the benchmark holds is in this notebook.

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.biased-noise.yaml
# config/benchmark/bp.data.benchmark.biased-noise.yaml

name: biased-noise
label: Biased-noise study
notes: >-
  An optional study, kept apart from the core benchmark (docs/CHALLENGE.md, What is
  fixed and what we choose): if a processor's errors were mostly phase flips and its
  CX kept them so, would an XX check catch nearly every error on the pair? Each graph
  runs the organisers' baseline, move_chain alone, and move_chain with bus_xx, under
  three pauli profiles that differ only in the native two-qubit gate and the noise
  bias: pauli-cx-biased (CX, eta 100), pauli-cz-biased (CZ, eta 100) and
  pauli-cx-unbiased (CX, eta 0.5). On the contemporary and hyperbolic graphs bus_xx
  runs on a spare path and reads only at the end; ibm2016 has no spare path, so there
  it runs on the path, reads each qubit before reusing it, and needs the mid-circuit
  measurement channel, set to 0.01 as one point of the range it is to be swept over.
  The profiles have no readout error, decay or durations, so these numbers describe
  gate noise alone, on one finite model each. 20,000 shots per basis, as in the core.
shots: 20000
base_seed: 20261116
runs:
  # Under pauli-cx-biased.
  - {run: baseline-ibm2016-cx-biased, circuit: baseline-ibm2016, noise: pauli-cx-biased}
  - {run: baseline-contemporary-cx-biased, circuit: baseline-contemporary, noise: pauli-cx-biased}
  - {run: baseline-hyperbolic-cx-biased, circuit: baseline-hyperbolic, noise: pauli-cx-biased}
  - {run: move-ibm2016-cx-biased, circuit: move-chain-ibm2016, noise: pauli-cx-biased}
  - {run: move-contemporary-cx-biased, circuit: move-chain-contemporary, noise: pauli-cx-biased}
  - {run: move-hyperbolic-cx-biased, circuit: move-chain-hyperbolic, noise: pauli-cx-biased}
  - {run: path-xx-ibm2016-cx-biased, circuit: move-chain-path-xx-ibm2016, noise: pauli-cx-biased, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary-cx-biased, circuit: move-chain-spare-xx-contemporary, noise: pauli-cx-biased}
  - {run: spare-xx-hyperbolic-cx-biased, circuit: move-chain-spare-xx-hyperbolic, noise: pauli-cx-biased}
  # Under pauli-cz-biased.
  - {run: baseline-ibm2016-cz-biased, circuit: baseline-ibm2016, noise: pauli-cz-biased}
  - {run: baseline-contemporary-cz-biased, circuit: baseline-contemporary, noise: pauli-cz-biased}
  - {run: baseline-hyperbolic-cz-biased, circuit: baseline-hyperbolic, noise: pauli-cz-biased}
  - {run: move-ibm2016-cz-biased, circuit: move-chain-ibm2016, noise: pauli-cz-biased}
  - {run: move-contemporary-cz-biased, circuit: move-chain-contemporary, noise: pauli-cz-biased}
  - {run: move-hyperbolic-cz-biased, circuit: move-chain-hyperbolic, noise: pauli-cz-biased}
  - {run: path-xx-ibm2016-cz-biased, circuit: move-chain-path-xx-ibm2016, noise: pauli-cz-biased, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary-cz-biased, circuit: move-chain-spare-xx-contemporary, noise: pauli-cz-biased}
  - {run: spare-xx-hyperbolic-cz-biased, circuit: move-chain-spare-xx-hyperbolic, noise: pauli-cz-biased}
  # Under pauli-cx-unbiased.
  - {run: baseline-ibm2016-cx-unbiased, circuit: baseline-ibm2016, noise: pauli-cx-unbiased}
  - {run: baseline-contemporary-cx-unbiased, circuit: baseline-contemporary, noise: pauli-cx-unbiased}
  - {run: baseline-hyperbolic-cx-unbiased, circuit: baseline-hyperbolic, noise: pauli-cx-unbiased}
  - {run: move-ibm2016-cx-unbiased, circuit: move-chain-ibm2016, noise: pauli-cx-unbiased}
  - {run: move-contemporary-cx-unbiased, circuit: move-chain-contemporary, noise: pauli-cx-unbiased}
  - {run: move-hyperbolic-cx-unbiased, circuit: move-chain-hyperbolic, noise: pauli-cx-unbiased}
  - {run: path-xx-ibm2016-cx-unbiased, circuit: move-chain-path-xx-ibm2016, noise: pauli-cx-unbiased, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary-cx-unbiased, circuit: move-chain-spare-xx-contemporary, noise: pauli-cx-unbiased}
  - {run: spare-xx-hyperbolic-cx-unbiased, circuit: move-chain-spare-xx-hyperbolic, noise: pauli-cx-unbiased}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-contemporary.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-contemporary.yaml

name: deliverable-2026-11-17-extended-contemporary
label: Contemporary, the protected case beside its alternatives
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): on the contemporary graph, the deliverable's protected case,
  case 6, beside the circuits that differ from it by one step, so each comparison
  shows what one change buys. The rungs, each a builder call along the baseline's path
  [0, 1, 2, 3, 7]: the organisers' baseline; the move chain, which moves the pair with
  two CX gates a hop instead of a SWAP's three; ghz_measure, the textbook GHZ
  transport, which reads the interior qubits in X and owes a Z on the first port for
  each 1; ghz_uncompute, which empties the interior with gates instead; each of the
  two empty transports followed by a vacancy read at the end, where only
  ghz_uncompute's last gate makes the read catch a bit flip on the first port; and two
  XX checks on a spare path that shares no qubit with the pair's: the move-chain
  benchmark's case 6, the move chain with the check, and ghz_uncompute with its
  vacancy read followed by the check, which trades discarded shots for F. Both XX
  buses on this graph pass through its five-edge qubit, so their numbers are without
  the planned degree penalty (docs/DESIGN.md, section 4). Every run takes the
  deliverable's noise profile, shots and base seed, so the runs it shares with the
  deliverable repeat its numbers exactly; the runs it shares with the move-chain
  benchmark, which takes 20,000 shots, agree with them within their errors.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-contemporary, circuit: baseline-contemporary, noise: boston-2026}
  # The transports alone, with no check.
  - {run: move-contemporary, circuit: move-chain-contemporary, noise: boston-2026}
  - {run: ghz-measure-contemporary, circuit: ghz-measure-contemporary, noise: boston-2026}
  - {run: ghz-contemporary, circuit: ghz-uncompute-contemporary, noise: boston-2026}
  # A vacancy read at the end after each transport that empties the path; the second is the deliverable's protected case.
  - {run: move-vacancy-contemporary, circuit: move-chain-vacancy-contemporary, noise: boston-2026}
  - {run: ghz-vacancy-contemporary, circuit: ghz-uncompute-vacancy-contemporary, noise: boston-2026}
  # XX checks on a spare path: the move-chain benchmark's case 6, then the vacancy read followed by the check.
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: ghz-vacancy-xx-contemporary, circuit: ghz-uncompute-vacancy-spare-xx-contemporary, noise: boston-2026}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-hyperbolic.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-hyperbolic.yaml

name: deliverable-2026-11-17-extended-hyperbolic
label: Hyperbolic-inspired, the protected case beside its alternatives
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): on the hyperbolic-inspired graph, the deliverable's protected
  case, case 9, beside the circuits that differ from it by one step, so each
  comparison shows what one change buys. The rungs, each a builder call along the
  baseline's path [0, 1, 9, 12]: the organisers' baseline; the move chain, which moves
  the pair with two CX gates a hop instead of a SWAP's three; ghz_measure, the
  textbook GHZ transport, which reads the interior qubits in X and owes a Z on the
  first port for each 1; ghz_uncompute, which empties the interior with gates instead;
  each of the two empty transports followed by a vacancy read at the end, where only
  ghz_uncompute's last gate makes the read catch a bit flip on the first port; and two
  XX checks on a spare path that shares no qubit with the pair's: the move-chain
  benchmark's case 9, the move chain with the check, and ghz_uncompute with its
  vacancy read followed by the check, which trades discarded shots for F. Every run
  takes the deliverable's noise profile, shots and base seed, so the runs it shares
  with the deliverable repeat its numbers exactly; the runs it shares with the
  move-chain benchmark, which takes 20,000 shots, agree with them within their errors.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
  # The transports alone, with no check.
  - {run: move-hyperbolic, circuit: move-chain-hyperbolic, noise: boston-2026}
  - {run: ghz-measure-hyperbolic, circuit: ghz-measure-hyperbolic, noise: boston-2026}
  - {run: ghz-hyperbolic, circuit: ghz-uncompute-hyperbolic, noise: boston-2026}
  # A vacancy read at the end after each transport that empties the path; the second is the deliverable's protected case.
  - {run: move-vacancy-hyperbolic, circuit: move-chain-vacancy-hyperbolic, noise: boston-2026}
  - {run: ghz-vacancy-hyperbolic, circuit: ghz-uncompute-vacancy-hyperbolic, noise: boston-2026}
  # XX checks on a spare path: the move-chain benchmark's case 9, then the vacancy read followed by the check.
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
  - {run: ghz-vacancy-xx-hyperbolic, circuit: ghz-uncompute-vacancy-spare-xx-hyperbolic, noise: boston-2026}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-ibm2016.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-ibm2016.yaml

name: deliverable-2026-11-17-extended-ibm2016
label: 2016-inspired, the protected case beside its alternatives
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): on the 2016-inspired graph, the deliverable's protected case,
  case 3, beside the circuits that differ from it by one step, so each comparison
  shows what one change buys. The rungs, each a builder call along the baseline's path
  [0, 3, 4]: the organisers' baseline; the move chain, which moves the pair with two
  CX gates a hop instead of a SWAP's three; ghz_measure, the textbook GHZ transport,
  which reads the interior qubits in X and owes a Z on the first port for each 1;
  ghz_uncompute, which empties the interior with gates instead; each of the two empty
  transports followed by a vacancy read at the end, where only ghz_uncompute's last
  gate makes the read catch a bit flip on the first port; and the move-chain
  benchmark's case 3, the move chain with an XX check on the pair's own path, which
  reads a qubit mid-circuit and runs with the mid-circuit measurement error of 0.01 it
  was reported with. The graph has no spare path, so no rung adds an XX check to the
  vacancy read. Every run takes the deliverable's noise profile, shots and base seed,
  so the runs it shares with the deliverable repeat its numbers exactly; the runs it
  shares with the move-chain benchmark, which takes 20,000 shots, agree with them
  within their errors.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  # The transports alone, with no check.
  - {run: move-ibm2016, circuit: move-chain-ibm2016, noise: boston-2026}
  - {run: ghz-measure-ibm2016, circuit: ghz-measure-ibm2016, noise: boston-2026}
  - {run: ghz-ibm2016, circuit: ghz-uncompute-ibm2016, noise: boston-2026}
  # A vacancy read at the end after each transport that empties the path; the second is the deliverable's protected case.
  - {run: move-vacancy-ibm2016, circuit: move-chain-vacancy-ibm2016, noise: boston-2026}
  - {run: ghz-vacancy-ibm2016, circuit: ghz-uncompute-vacancy-ibm2016, noise: boston-2026}
  # The move-chain benchmark's case 3: the move chain with its XX check on the path, read mid-circuit.
  - {run: path-xx-ibm2016, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17.yaml

name: deliverable-2026-11-17
label: GHZ disentangling with a vacancy read, the nine cases
notes: >-
  The challenge's nine cases (docs/CHALLENGE.md, The nine core cases) as the
  submission reports them, under boston-2026 as in the core benchmark, so the three
  graphs differ only in connectivity. Cases 1, 2, 4, 5, 7 and 8 are the organisers'
  baseline, exactly as the core benchmark runs it. Cases 3, 6 and 9 are one protection
  strategy, the same builder steps on every graph: ghz_uncompute makes a GHZ state
  along the pair's path, then disentangles the path's interior qubits with gates
  controlled from the first port, leaving them in |0>; vacancy then reads them at the
  end of the circuit, and postselection discards each shot where any reads 1. The read
  catches a bit flip on the interior qubits and on the first port, the qubit that has
  waited longest, since the last gate copies it onto the path; it cannot see a phase
  flip. No qubit is read mid-circuit, so no case needs the mid-circuit measurement
  channel. The same transport also runs without the read, filling no case, so the
  table separates what the transport buys from what the read buys. Each graph's
  extended benchmark, deliverable-2026-11-17-extended-<processor>, sets these runs
  beside the alternatives. Every run takes 100,000 shots per basis, five times the
  core's 20,000, so F's standard error is about 0.0003 and a difference of 0.001
  between two runs stands at more than two combined standard errors; the baseline
  cases therefore agree with the core's within their errors rather than repeating its
  numbers.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: baseline-contemporary, circuit: baseline-contemporary, noise: boston-2026}
  - {run: baseline-hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
  # The transport alone, filling no case.
  - {run: ghz-ibm2016, circuit: ghz-uncompute-ibm2016, noise: boston-2026}
  - {run: ghz-contemporary, circuit: ghz-uncompute-contemporary, noise: boston-2026}
  - {run: ghz-hyperbolic, circuit: ghz-uncompute-hyperbolic, noise: boston-2026}
  # The protection: the transport with its vacancy read at the end.
  - {run: ghz-vacancy-ibm2016, circuit: ghz-uncompute-vacancy-ibm2016, noise: boston-2026}
  - {run: ghz-vacancy-contemporary, circuit: ghz-uncompute-vacancy-contemporary, noise: boston-2026}
  - {run: ghz-vacancy-hyperbolic, circuit: ghz-uncompute-vacancy-hyperbolic, noise: boston-2026}
cases:
  - {case: 1, run: baseline-ibm2016, simulation: ideal}
  - {case: 2, run: baseline-ibm2016, simulation: noisy}
  - {case: 3, run: ghz-vacancy-ibm2016, simulation: noisy}
  - {case: 4, run: baseline-contemporary, simulation: ideal}
  - {case: 5, run: baseline-contemporary, simulation: noisy}
  - {case: 6, run: ghz-vacancy-contemporary, simulation: noisy}
  - {case: 7, run: baseline-hyperbolic, simulation: ideal}
  - {case: 8, run: baseline-hyperbolic, simulation: noisy}
  - {case: 9, run: ghz-vacancy-hyperbolic, simulation: noisy}

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-measure-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-measure-contemporary.yaml

name: ghz-measure-contemporary
label: ghz_measure, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks: []
    frame_z:
    - frame_z_ghz[0]
    - frame_z_ghz[1]
    - frame_z_ghz[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_measure(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] frame_z_ghz;
  qubit[9] q;
  h q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[1];
  cx q[1], q[0];
  h q[1];
  frame_z_ghz[0] = measure q[1];
  h q[2];
  frame_z_ghz[1] = measure q[2];
  h q[3];
  frame_z_ghz[2] = measure q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-measure-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-measure-hyperbolic.yaml

name: ghz-measure-hyperbolic
label: ghz_measure, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks: []
    frame_z:
    - frame_z_ghz[0]
    - frame_z_ghz[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_measure(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] frame_z_ghz;
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[0];
  h q[1];
  frame_z_ghz[0] = measure q[1];
  h q[9];
  frame_z_ghz[1] = measure q[9];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-measure-ibm2016.yaml
# config/circuit/bp.data.circuit.ghz-measure-ibm2016.yaml

name: ghz-measure-ibm2016
label: ghz_measure, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks: []
    frame_z:
    - frame_z_ghz[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: ghz_measure(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] frame_z_ghz;
  qubit[5] q;
  h q[3];
  cx q[3], q[4];
  cx q[3], q[0];
  h q[3];
  frame_z_ghz[0] = measure q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-contemporary.yaml

name: ghz-uncompute-contemporary
label: ghz_uncompute, contemporary
processor: contemporary
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_uncompute().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[3];
  cx q[1], q[2];
  cx q[0], q[1];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-hyperbolic.yaml

name: ghz-uncompute-hyperbolic
label: ghz_uncompute, hyperbolic
processor: hyperbolic
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_uncompute().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[9];
  cx q[0], q[1];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-ibm2016.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-ibm2016.yaml

name: ghz-uncompute-ibm2016
label: ghz_uncompute, ibm2016
processor: ibm2016
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: ghz_uncompute().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[0], q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-contemporary.yaml

name: ghz-uncompute-vacancy-contemporary
label: ghz_uncompute then vacancy at the end, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_vac[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_uncompute(); then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_vac;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[3];
  cx q[1], q[2];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[2];
  check_vac[2] = measure q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-hyperbolic.yaml

name: ghz-uncompute-vacancy-hyperbolic
label: ghz_uncompute then vacancy at the end, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_uncompute();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_vac;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[9];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[9];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-vacancy-contemporary.yaml
# config/circuit/bp.data.circuit.move-chain-vacancy-contemporary.yaml

name: move-chain-vacancy-contemporary
label: move_chain then vacancy at the end, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_vac[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  move_chain(); then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_vac;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[2];
  check_vac[2] = measure q[3];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-vacancy-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-chain-vacancy-hyperbolic.yaml

name: move-chain-vacancy-hyperbolic
label: move_chain then vacancy at the end, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: move_chain();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_vac;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[9];

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-vacancy-ibm2016.yaml
# config/circuit/bp.data.circuit.move-chain-vacancy-ibm2016.yaml

name: move-chain-vacancy-ibm2016
label: move_chain then vacancy at the end, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: move_chain();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_vac;
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[4], q[3];
  check_vac[0] = measure q[3];

In [ ]:
%%repo_file config/noise/bp.data.noise.burlington-2020.yaml
# config/noise/bp.data.noise.burlington-2020.yaml
#
# Medians of IBM's FakeBurlingtonV2 snapshot, a five-qubit T-graph processor.
# Validated by config/bp.schema.noise.yaml; tests/test_config.py pins every value
# to the snapshot and every comment to the schema.

name: burlington-2020
label: FakeBurlingtonV2 medians, calibrated 2020-06-11
kind: snapshot
reference: qiskit-fake-provider
snapshot:
  class: FakeBurlingtonV2
  package: qiskit-ibm-runtime==0.51.0
  calibrated: "2020-06-11"
  properties: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/burlington/props_burlington.json
  configuration: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/burlington/conf_burlington.json
notes: >-
  Medians over the FakeBurlingtonV2 snapshot's 5 qubits and 8 directed CX
  couplers, as qiskit-ibm-runtime 0.51.0 pins them; it is the earliest calibration
  among the snapshots with the T graph. The snapshot's u1, u2 and u3 gates are
  expressed as rz, sx and x: u2 is one pulse and u3 two. The snapshot has no
  measurement duration, so it is borrowed and marked below. It has no reset data,
  so a circuit that resets cannot run on this profile until a value is chosen.
# Time step in seconds: the hardware's clock tick. Null when the snapshot records none.
time_step: null
qubit:
  # T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1).
  t1_relaxation_time: 82.62697735235098e-6
  # T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1.
  t2_dephasing_time: 93.19832447669017e-6
  readout_error:
    # Readout error: the probability of reading 1 when the qubit was 0.
    reads_1_when_0: 0.023
    # Readout error: the probability of reading 0 when the qubit was 1.
    reads_0_when_1: 0.041
# Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds.
gates:
  cx: {qubits: 2, error_probability: 0.016097735173324107, duration: 389.3333333333333e-9}
  sx: {qubits: 1, error_probability: 0.0005738102719058851, duration: 53.33333333333333e-9, from_snapshot_gate: u2}
  x: {qubits: 1, error_probability: 0.0011472912855835515, duration: 106.66666666666666e-9, from_snapshot_gate: u3}
  rz: {qubits: 1, error_probability: 0.0, duration: 0.0, from_snapshot_gate: u1}
# Measurement: its duration in seconds. Its errors are the readout errors.
measure: {duration: 5.8133333333333335e-6}
borrowed:
  - field: measure.duration
    value: 5.8133333333333335e-6
    source: "FakeOurenseV2 median (calibrated 2021-01-20), qiskit-ibm-runtime==0.51.0"
    url: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/ourense/props_ourense.json
    why: >-
      FakeBurlingtonV2 records no measurement duration. The earliest T-graph snapshots
      that record one, FakeOurenseV2, FakeValenciaV2 and FakeVigoV2 (all calibrated
      2021-01-20), all give this value.

In [ ]:
%%repo_file config/noise/bp.data.noise.fez-2025.yaml
# config/noise/bp.data.noise.fez-2025.yaml
#
# Medians of IBM's FakeFez snapshot (Heron r2). Validated by
# config/bp.schema.noise.yaml; tests/test_config.py pins every value to the
# snapshot and every comment to the schema.

name: fez-2025
label: FakeFez medians, calibrated 2025-02-26
kind: snapshot
reference: qiskit-fake-provider
snapshot:
  class: FakeFez
  package: qiskit-ibm-runtime==0.51.0
  calibrated: "2025-02-26"
  properties: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/fez/props_fez.json
  configuration: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/fez/conf_fez.json
notes: >-
  Medians over the FakeFez snapshot's 156 qubits and 352 directed CZ couplers, as
  qiskit-ibm-runtime 0.51.0 pins them. Every qubit and coupler of a processor
  graph takes these values.
# Time step in seconds: the hardware's clock tick. Null when the snapshot records none.
time_step: 4.0e-9
qubit:
  # T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1).
  t1_relaxation_time: 144.85526609546147e-6
  # T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1.
  t2_dephasing_time: 87.95068681159162e-6
  readout_error:
    # Readout error: the probability of reading 1 when the qubit was 0.
    reads_1_when_0: 0.00390625
    # Readout error: the probability of reading 0 when the qubit was 1.
    reads_0_when_1: 0.011474609375
# Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds.
gates:
  cz: {qubits: 2, error_probability: 0.0039033828523107883, duration: 84.0e-9}
  sx: {qubits: 1, error_probability: 0.00022868757107977525, duration: 24.0e-9}
  x: {qubits: 1, error_probability: 0.00022868757107977525, duration: 24.0e-9}
  rz: {qubits: 1, error_probability: 0.0, duration: 0.0}
# Measurement: its duration in seconds. Its errors are the readout errors.
measure: {duration: 1.56e-6}
# Reset to 0: its duration in seconds.
reset: {duration: 1.584e-6}
borrowed: []

In [ ]:
%%repo_file config/noise/bp.data.noise.organiser-cx2.yaml
# config/noise/bp.data.noise.organiser-cx2.yaml
#
# The organisers' noise from their example notebook, used as a cross-check that
# reproduces their recorded fidelities, never as the core noise model. Validated
# by config/bp.schema.noise.yaml; tests/test_config.py pins every comment to the
# schema.

name: organiser-cx2
label: The organisers' 2% CX depolarising noise
kind: depolarising
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 4 (the noise model cell): a two-qubit
  depolarising channel of strength 0.02 after every CX, every other gate ideal, no
  readout error and no idle decay. Their recipe uses only H and CX; sdg is added,
  ideal like H, so a run can rotate the ports into the YY basis, which the organisers
  read from the density matrix instead of measuring.
depolarising:
  # The gate each depolarising channel follows.
  after_gate: cx
  # Depolarising strength: with this probability, the gate's qubits are left in a fully random state.
  probability: 0.02
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cx, sdg]

In [ ]:
%%repo_file config/noise/bp.data.noise.pauli-cx-biased.yaml
# config/noise/bp.data.noise.pauli-cx-biased.yaml
#
# A Pauli noise profile for the biased-noise study, kept apart from the core
# benchmark. Validated by config/bp.schema.noise.yaml; tests/test_config.py pins
# every comment to the schema.

name: pauli-cx-biased
label: Pauli noise biased toward phase flips (eta 100), CX native
kind: pauli
reference: puri-2020-bias-preserving-cx
notes: >-
  A what-if for the biased-noise study, never a core model. After every CX, each of its
  two qubits independently takes a Pauli error with probability 0.00942, split by the
  noise bias eta = p_Z / (p_X + p_Y) as p_X = p_Y = p / (2 (eta + 1)). The probability
  makes a two-qubit gate's total error, 1 - (1 - p)^2, about 0.01875, match
  organiser-cx2's (15/16 of its 0.02). With CX native, a phase flip stays a phase flip
  through the gate, the bias-preserving CX that Puri et al. (2020) propose for
  stabilised cat qubits. Only the CX carries an error, so this is the case where the
  bias reaches the pair intact. Every other gate is ideal, with no readout error, no
  decay and no durations, so a run is not scheduled.
pauli:
  # The gate each Pauli error follows, on every qubit it acts on.
  after_gate: cx
  # The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit.
  probability: 0.00942
  # Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is.
  bias: 100
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cx, sdg]

In [ ]:
%%repo_file config/noise/bp.data.noise.pauli-cx-unbiased.yaml
# config/noise/bp.data.noise.pauli-cx-unbiased.yaml
#
# A Pauli noise profile for the biased-noise study, kept apart from the core
# benchmark. Validated by config/bp.schema.noise.yaml; tests/test_config.py pins
# every comment to the schema.

name: pauli-cx-unbiased
label: Unbiased Pauli noise (eta 0.5), CX native
kind: pauli
reference: puri-2020-bias-preserving-cx
notes: >-
  A what-if for the biased-noise study, never a core model. After every CX, each of its
  two qubits independently takes a Pauli error with probability 0.00942, split by the
  noise bias eta = p_Z / (p_X + p_Y) as p_X = p_Y = p / (2 (eta + 1)). The probability
  makes a two-qubit gate's total error, 1 - (1 - p)^2, about 0.01875, match
  organiser-cx2's (15/16 of its 0.02). The control: the same channel structure as
  pauli-cx-biased with X, Y and Z equally likely, which on each qubit is a one-qubit
  depolarising channel, so a difference from pauli-cx-biased comes from the noise bias
  alone. Every other gate is ideal, with no readout error, no decay and no durations, so
  a run is not scheduled.
pauli:
  # The gate each Pauli error follows, on every qubit it acts on.
  after_gate: cx
  # The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit.
  probability: 0.00942
  # Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is.
  bias: 0.5
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cx, sdg]

In [ ]:
%%repo_file config/noise/bp.data.noise.pauli-cz-biased.yaml
# config/noise/bp.data.noise.pauli-cz-biased.yaml
#
# A Pauli noise profile for the biased-noise study, kept apart from the core
# benchmark. Validated by config/bp.schema.noise.yaml; tests/test_config.py pins
# every comment to the schema.

name: pauli-cz-biased
label: Pauli noise biased toward phase flips (eta 100), CZ native
kind: pauli
reference: puri-2020-bias-preserving-cx
notes: >-
  A what-if for the biased-noise study, never a core model. After every CZ, each of its
  two qubits independently takes a Pauli error with probability 0.00942, split by the
  noise bias eta = p_Z / (p_X + p_Y) as p_X = p_Y = p / (2 (eta + 1)). The probability
  makes a two-qubit gate's total error, 1 - (1 - p)^2, about 0.01875, match
  organiser-cx2's (15/16 of its 0.02). With CZ native, a CX compiles as H CZ H, and the
  second H turns a phase flip on the CX's target into a bit flip, so the bias is partly
  lost. Same error as pauli-cx-biased, so the two differ only in the native two-qubit
  gate. Every other gate is ideal, with no readout error, no decay and no durations, so
  a run is not scheduled.
pauli:
  # The gate each Pauli error follows, on every qubit it acts on.
  after_gate: cz
  # The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit.
  probability: 0.00942
  # Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is.
  bias: 100
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cz, sdg]

<a id="sD"></a>

## Appendix D. Sources and AI use

**Sources.** Every source we rely on is an entry in [`docs/bp.data.references.yaml`](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/bp.data.references.yaml), with its link and what we take from it. The repository holds the full benchmark: [Rickory2021/bell-transport-bench](https://github.com/Rickory2021/bell-transport-bench/tree/submission).

**AI use.** We used AI assistants, including Claude, for research summaries, design discussion, and drafting code and documentation. The team reviewed and ran everything in this notebook and can explain it.

[Back to the top](#top)